# Baseline MobileNetV2-UNet trên Kaggle — 1 GPU, train và đánh giá đầy đủ

Notebook tự chứa toàn bộ pipeline và đọc trực tiếp dataset `tinlngnguyn/skin-lesion-segmentation-hus` đã Attach tại `/kaggle/input`. Training chính thức được chia 3 fold; LR Finder chỉ dùng train subset, augmentation chỉ dùng cho train subset và checkpoint được chọn bằng Best Dice trên fold holdout sạch.

Sau khi cả 3 fold đã được khóa, notebook đánh giá từng fold model và equal-weight ensemble trên **official validation**, rồi đánh giá single model đã chọn từ CV và ensemble trên **official test**. Validation/test tuyệt đối không tham gia LR Finder, train hoặc chọn checkpoint.

> Trên Kaggle: `Add Input` dataset `tinlngnguyn/skin-lesion-segmentation-hus`, chọn Accelerator GPU (một GPU là đủ) và bật Internet nếu cache MobileNetV2 ImageNet chưa có. Artifact được ghi vào `/kaggle/working/mobilenetv2_unet_kaggle_1gpu`.

# MobileNetV2-UNet – Phân đoạn vùng tổn thương da

## 1. Tổng quan

Notebook này triển khai quy trình huấn luyện và đánh giá mô hình học sâu **MobileNetV2-UNet** cho bài toán phân đoạn vùng tổn thương da từ ảnh dermoscopic (ảnh soi da).

Mục tiêu chính là xây dựng mô hình có khả năng tự động xác định và phân đoạn vùng tổn thương trên ảnh đầu vào.

Notebook sử dụng kiến trúc U-Net với MobileNetV2 làm bộ mã hóa (Encoder), kết hợp phương pháp kiểm định chéo 3-Fold Cross-Validation để huấn luyện và đánh giá mô hình.

Ngoài ra, thực nghiệm so sánh hiệu quả giữa:
- **Single Best Model:** Mô hình đơn có kết quả Internal Holdout Dice cao nhất.
- **Equal-Weight Ensemble:** Mô hình kết hợp dự đoán từ cả ba mô hình được huấn luyện qua 3 Fold.

## 2. Dữ liệu

Bộ dữ liệu được sử dụng từ Kaggle:

- **Dataset:** `tinlngnguyn/skin-lesion-segmentation-hus`
- **Tập Training:** 2.594 ảnh.
- **Tập Validation:** 100 ảnh.
- **Tập Test:** 1.000 ảnh.

Mỗi mẫu dữ liệu bao gồm:
- Ảnh dermoscopic đầu vào.
- Ảnh mặt nạ phân đoạn tương ứng (Ground Truth Mask).

Thực nghiệm sử dụng các tập dữ liệu được chia sẵn theo bộ dữ liệu chính thức, không tự ý thay đổi cách phân chia giữa Training, Validation và Test.

## 3. Kiến trúc mô hình

Mô hình sử dụng kiến trúc **MobileNetV2-UNet**, kết hợp giữa mạng MobileNetV2 và kiến trúc U-Net.

Các thành phần chính:

- **Encoder – MobileNetV2:** Trích xuất đặc trưng từ ảnh đầu vào ở nhiều mức độ phân giải. Encoder được khởi tạo bằng trọng số pretrained trên ImageNet.
- **Decoder:** Khôi phục dần kích thước không gian của đặc trưng để tạo ra bản đồ phân đoạn.
- **Skip Connections:** Kết nối đặc trưng từ Encoder sang Decoder nhằm giữ lại thông tin không gian và chi tiết của vùng tổn thương.
- **Output Layer:** Sử dụng lớp tích chập cuối cùng để tạo đầu ra một kênh (raw logits), phục vụ bài toán phân đoạn nhị phân.

Đầu ra của mô hình là bản đồ dự đoán vùng tổn thương, có cùng kích thước không gian với ảnh đầu vào.

## 4. Tiền xử lý dữ liệu và tăng cường dữ liệu

### 4.1. Tiền xử lý

Quy trình tiền xử lý bao gồm:

- Thay đổi kích thước ảnh và mask về kích thước đầu vào được lựa chọn.
- Cân bằng màu sắc bằng phương pháp Gray World.
- Chuyển đổi ảnh và mask thành Tensor.
- Chuẩn hóa ảnh theo ImageNet Mean và Standard Deviation trên GPU.

### 4.2. Tăng cường dữ liệu (Data Augmentation)

Các phép biến đổi được áp dụng trên tập Training nhằm tăng tính đa dạng của dữ liệu:

- D4 transformations.
- Biến đổi Affine với góc xoay và tỷ lệ phóng đại được cấu hình.
- Điều chỉnh độ sáng và độ tương phản.

**Lưu ý:** Data Augmentation chỉ được áp dụng cho dữ liệu huấn luyện. Các tập Internal Holdout, Official Validation và Official Test không sử dụng tăng cường dữ liệu ngẫu nhiên khi đánh giá.

## 5. Quy trình huấn luyện

Notebook sử dụng phương pháp **3-Fold Cross-Validation** trên tập Training chính thức.

Dữ liệu Training được chia thành ba phần (Fold). Trong mỗi lần huấn luyện:

- Hai Fold được sử dụng để huấn luyện mô hình.
- Một Fold được giữ lại làm tập Internal Holdout để đánh giá mô hình.

Quá trình này được lặp lại ba lần để mỗi Fold đều được sử dụng làm tập Holdout một lần.

### 5.1. Hai giai đoạn huấn luyện

Tại mỗi Fold, mô hình được huấn luyện ở hai kích thước ảnh: 128 × 128 và 256 × 256.

Mỗi kích thước ảnh gồm hai giai đoạn:

**Giai đoạn 1 – Frozen:**
- Đóng băng các tham số Encoder theo chiến lược được thiết lập.
- Cho phép Decoder học các đặc trưng phục vụ phân đoạn.
- Giữ lại các tham số BatchNorm được cho phép huấn luyện theo cấu hình.

**Giai đoạn 2 – Unfrozen:**
- Mở các tham số Encoder để tiếp tục tinh chỉnh mô hình.
- Các lớp BatchNorm được giữ ở chế độ đánh giá nhằm hạn chế cập nhật thống kê trong quá trình huấn luyện.

### 5.2. Cấu hình huấn luyện

Các thành phần chính được sử dụng:

- **Loss Function:** BCEWithLogitsLoss.
- **Optimizer:** Adam.
- **Learning Rate Finder:** Tìm kiếm tốc độ học phù hợp.
- **Learning Rate Scheduler:** Slanted Triangular Learning Rate (STLR).
- **Mixed Precision Training (AMP):** Giảm mức sử dụng bộ nhớ GPU và hỗ trợ tăng tốc huấn luyện khi được bật.
- **Gradient Clipping:** Giới hạn độ lớn gradient nhằm hỗ trợ ổn định quá trình tối ưu.
- **Checkpoint Resumption:** Hỗ trợ tiếp tục huấn luyện từ checkpoint đã lưu.

Mô hình được đánh giá trên tập Internal Holdout sau mỗi giai đoạn. Trạng thái mô hình tốt nhất trong từng giai đoạn được lựa chọn dựa trên Dice Score.

## 6. Lựa chọn mô hình và kết hợp Ensemble

Sau khi hoàn thành huấn luyện ba Fold, notebook triển khai hai phương pháp sử dụng mô hình.

### 6.1. Single Best Model

Lựa chọn mô hình có điểm Internal Holdout Dice cao nhất trong ba Fold.

Mô hình được lựa chọn sẽ được lưu thành một checkpoint riêng để sử dụng trong giai đoạn đánh giá chính thức.

### 6.2. Equal-Weight Ensemble

Kết hợp dự đoán từ cả ba mô hình Fold bằng phương pháp lấy trung bình xác suất dự đoán.

Tại mỗi pixel, xác suất dự đoán của ba mô hình được tính trung bình để tạo ra bản đồ xác suất cuối cùng.

Sau đó, sử dụng ngưỡng dự đoán được thiết lập để chuyển bản đồ xác suất thành mask nhị phân.

Mục tiêu là đánh giá liệu việc kết hợp nhiều mô hình có giúp cải thiện hiệu quả phân đoạn so với việc chỉ sử dụng một mô hình đơn tốt nhất hay không.

## 7. Đánh giá mô hình

Các mô hình được đánh giá trên hai tập dữ liệu chính thức: Validation và Test.

Các chỉ số đánh giá bao gồm:

- **Dice Score:** Đo mức độ chồng lấp giữa vùng tổn thương được dự đoán và vùng tổn thương thực tế.
- **IoU (Intersection over Union):** Đo tỷ lệ giữa phần giao và phần hợp của hai vùng phân đoạn.
- **Sensitivity (Recall):** Đo tỷ lệ pixel thuộc vùng tổn thương thực tế được mô hình phát hiện chính xác.
- **HD95 (95th Percentile Hausdorff Distance):** Đánh giá khoảng cách giữa biên của vùng dự đoán và biên của vùng thực tế. Chỉ số được báo cáo theo đơn vị pixel tại độ phân giải đầu vào cuối cùng.

### 7.1. Đánh giá trên Validation

Thực hiện đánh giá:
- Mô hình Fold 1.
- Mô hình Fold 2.
- Mô hình Fold 3.
- Equal-Weight Ensemble.

### 7.2. Đánh giá trên Test

Thực hiện đánh giá và so sánh:
- Single Best Model.
- Equal-Weight Ensemble.

Tập Validation và Test chính thức không được sử dụng để cập nhật trọng số mô hình hoặc lựa chọn mô hình trong quá trình huấn luyện.

## 8. Kết quả đầu ra

Notebook lưu trữ các kết quả thực nghiệm và checkpoint để phục vụ phân tích, so sánh và tái sử dụng.

Các tệp đầu ra chính bao gồm:

| Tệp | Nội dung |
|---|---|
| `config.json` | Cấu hình và các tham số của thực nghiệm |
| `fold_1_history.csv`, `fold_2_history.csv`, `fold_3_history.csv` | Lịch sử huấn luyện và đánh giá của từng Fold |
| `fold_holdout_dice.csv` | Điểm Internal Holdout Dice của ba Fold |
| `official_validation_metrics.csv` | Kết quả đánh giá trên tập Validation |
| `official_test_metrics.csv` | Kết quả đánh giá trên tập Test |
| `summary.json` | Tổng hợp cấu hình, thời gian huấn luyện, mô hình được chọn và kết quả đánh giá |
| `mobilenetv2_unet_best.pt` | Checkpoint của từng Fold |
| `mobilenetv2_unet_selected_single_best.pt` | Checkpoint của Single Best Model |

## 9. Mục tiêu thực nghiệm

Thực nghiệm hướng đến hai mục tiêu chính:

1. Xây dựng và đánh giá mô hình MobileNetV2-UNet trong bài toán phân đoạn vùng tổn thương da.
2. So sánh hiệu quả giữa phương pháp sử dụng một mô hình đơn tốt nhất và phương pháp kết hợp nhiều mô hình bằng Equal-Weight Ensemble.

Các kết quả thu được cũng được sử dụng để so sánh với những kiến trúc phân đoạn khác, chẳng hạn ResNet34-UNet, với điều kiện các mô hình được đánh giá theo cùng một quy trình thực nghiệm, bao gồm dữ liệu, cách chia tập, tiền xử lý, tăng cường dữ liệu và các chỉ số đánh giá.

In [4]:
# Kaggle image đã có PyTorch, torchvision, OpenCV và Albumentations.
# Không pip install giữa session để tránh thay đổi binary/package đang chạy.
print('Using the Kaggle runtime environment.')

Using the Kaggle runtime environment.


In [5]:
import gc
import hashlib
import json
import math
import os
import random
import time
from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from PIL import Image
from scipy import ndimage
from torch.utils.data import DataLoader, Dataset, get_worker_info
from torchvision import models
from tqdm.auto import tqdm

try:
    import psutil
except ImportError:
    psutil = None

# -------------------- Cấu hình baseline --------------------
DATASET_HANDLE = 'tinlngnguyn/skin-lesion-segmentation-hus'
DATA_ROOT = Path('/kaggle/input/datasets/tinlngnguyn/skin-lesion-segmentation-hus')
SPLIT_DIRECTORIES = {
    'train': ('ISIC2018_Task1-2_Training_Input', 'ISIC2018_Task1_Training_GroundTruth'),
    'validation': ('ISIC2018_Task1-2_Validation_Input', 'ISIC2018_Task1_Validation_GroundTruth'),
    'test': ('ISIC2018_Task1-2_Test_Input', 'ISIC2018_Task1_Test_GroundTruth'),
}
EXPECTED_COUNTS = {'train': 2594, 'validation': 100, 'test': 1000}
SEED = 42 #Seed là giá trị khởi tạo cho các thao tác ngẫu nhiên.
N_FOLDS = 3 
SIZES = (128, 256) # Ảnh sẽ được resize thành hai kích thước: 128 × 128 pixels. và 256 × 256 pixels.
FROZEN_EPOCHS = 30  # 30 epoch đầu: Chỉ dạy Decoder cách phân vùng ảnh, giữ nguyên Encoder.
UNFROZEN_EPOCHS = 30 # 30 epoch tiếp: Cho cả Encoder và Decoder cùng học để cải thiện kết quả.
PER_GPU_BATCH_BY_SIZE = {128: 16, 256: 8} # Batch size = 16: Mỗi lần GPU xử lý 16 ảnh., Batch size = 8: Mỗi lần GPU xử lý 8 ảnh.
NUM_WORKERS = min(2, os.cpu_count() or 1) # NUM_WORKERS là số tiến trình phụ giúp CPU chuẩn bị ảnh cho GPU
PREFETCH_FACTOR = 1 # PREFETCH_FACTOR quyết định mỗi người chuẩn bị trước bao nhiêu Batch.
PIN_MEMORY = True
CACHE_RESIZED_IN_RAM = True
DROP_OS_PAGE_CACHE = True
SAVE_LAST_EVERY = 5
LRF_ITERS = 50
LR_MIN, LR_MAX, LRF_BETA = 1e-6, 1.0, 0.9
LR_SELECTION_FACTOR = 0.1
MAX_LR_FROZEN, MAX_LR_UNFROZEN = 3e-3, 1e-3
CUT_FRAC, STLR_RATIO = 0.1, 32.0
WEIGHT_DECAY = 0.0 # tham số điều chỉnh mức phạt đối với trọng số lớn, thường được sử dụng để hạn chế overfitting
GRAD_CLIP_NORM = 1.0 # Gradient là thông tin cho biết trọng số cần thay đổi theo hướng nào và mức độ ra sao.
PRED_THRESHOLD = 0.5 # ngưỡng chuyển kết quả dự đoán thành mặt nạ nhị phân
JACCARD_CUT = 0.65 # ngưỡng Jaccard (IoU) được cấu hình để phân loại kết quả theo mức đạt yêu cầu trong notebook
PRETRAINED = True # Lấy mô hình đã có trọng số học trước.
USE_CHANNELS_LAST = False
FP32_TRANSPOSED_CONV = False  # MobileNetV2 decoder dùng bilinear upsample, không dùng ConvTranspose2d
USE_FUSED_ADAM = False
RESUME = True
FAST_DEV_RUN = False # chế độ chạy thử nhanh.
PROTOCOL_VERSION = 3

if FAST_DEV_RUN:
    SIZES = (128,)
    FROZEN_EPOCHS = UNFROZEN_EPOCHS = 1
    LRF_ITERS = 8

if not hasattr(A, 'D4'): # D4 là một phép biến đổi hình học dành cho ảnh vuông, bao gồm các cách xoay và lật ảnh.
    raise RuntimeError('Albumentations runtime quá cũ: cần A.D4 và A.Compose(seed=...).')
try:
    A.Compose([], seed=SEED) # A.Compose() là chức năng dùng để kết hợp nhiều phép biến đổi ảnh thành một chuỗi xử lý.
except TypeError as error:
    raise RuntimeError('Albumentations runtime không hỗ trợ Compose(seed=...).') from error

OUTPUT_ROOT = Path('/kaggle/working/mobilenetv2_unet_kaggle_1gpu')
RUN_NAME = 'fast_dev_v3' if FAST_DEV_RUN else 'full_v3'
RUN_DIR = OUTPUT_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
LOG_DIR = RUN_DIR / 'logs'
for directory in (RUN_DIR, CHECKPOINT_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

set_seed()
cv2.setNumThreads(0)
torch.set_float32_matmul_precision('high')
if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật Accelerator GPU trong Kaggle Settings.')

DEVICE = torch.device('cuda:0')
GPU_COUNT = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
BATCH_SIZE_BY_SIZE = dict(PER_GPU_BATCH_BY_SIZE)  # notebook luôn chỉ dùng cuda:0
USE_AMP = True
print({
    'device': str(DEVICE), 'gpus': GPU_NAMES, 'global_batch': BATCH_SIZE_BY_SIZE,
    'workers': NUM_WORKERS, 'channels_last': USE_CHANNELS_LAST,
    'fp32_upconv': FP32_TRANSPOSED_CONV, 'run_dir': str(RUN_DIR),
})

def memory_status():
    parts = []
    if psutil is not None:
        process = psutil.Process(os.getpid())
        info = process.memory_full_info()
        main_bytes = getattr(info, 'uss', info.rss)
        worker_bytes = 0
        for child in process.children(recursive=True):
            try:
                child_info = child.memory_full_info()
                worker_bytes += getattr(child_info, 'uss', child_info.rss)
            except psutil.Error:
                pass
        parts.extend([f'ram_main={main_bytes / 2**30:.2f}GiB',
                      f'ram_workers={worker_bytes / 2**30:.2f}GiB'])
    allocated = torch.cuda.memory_allocated(0)
    reserved = torch.cuda.memory_reserved(0)
    parts.extend([f'cuda_alloc={allocated / 2**30:.2f}GiB',
                  f'cuda_reserved={reserved / 2**30:.2f}GiB'])
    return ' '.join(parts)

{'device': 'cuda:0', 'gpus': ['Tesla T4', 'Tesla T4'], 'global_batch': {128: 16, 256: 8}, 'workers': 2, 'channels_last': False, 'fp32_upconv': False, 'run_dir': '/kaggle/working/mobilenetv2_unet_kaggle_1gpu/full_v3'}


In [6]:
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}

def find_populated_directory(directory_name, suffixes):
    outer = DATA_ROOT / directory_name
    candidates = [outer / directory_name, outer]
    if DATA_ROOT.is_dir():
        candidates.extend(path for path in DATA_ROOT.rglob(directory_name) if path.is_dir())
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen or not candidate.is_dir():
            continue
        seen.add(candidate)
        if any(path.is_file() and path.suffix.lower() in suffixes
               for path in candidate.iterdir()):
            return candidate
    raise FileNotFoundError(f'Không tìm thấy thư mục dữ liệu {directory_name} dưới {DATA_ROOT}')

def discover_records(split):
    image_name, mask_name = SPLIT_DIRECTORIES[split]
    image_dir = find_populated_directory(image_name, IMAGE_SUFFIXES)
    mask_dir = find_populated_directory(mask_name, {'.png'})
    image_paths = [path for path in image_dir.iterdir()
                   if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES]
    mask_paths = [path for path in mask_dir.iterdir()
                  if path.is_file() and path.suffix.lower() == '.png'
                  and path.stem.endswith('_segmentation')]
    if len({path.stem for path in image_paths}) != len(image_paths):
        raise RuntimeError(f'{split}: trùng image ID')
    image_map = {path.stem: path for path in image_paths}
    mask_map = {path.stem.removesuffix('_segmentation'): path for path in mask_paths}
    if len(mask_map) != len(mask_paths):
        raise RuntimeError(f'{split}: trùng mask ID')
    missing_masks = sorted(set(image_map) - set(mask_map))
    orphan_masks = sorted(set(mask_map) - set(image_map))
    if missing_masks or orphan_masks:
        raise RuntimeError(f'{split}: thiếu mask={missing_masks[:5]}, mask mồ côi={orphan_masks[:5]}')
    records = [
        {'image_id': image_id, 'image_path': image_map[image_id],
         'mask_path': mask_map[image_id]}
        for image_id in sorted(image_map)
    ]
    if len(records) != EXPECTED_COUNTS[split]:
        raise RuntimeError(f'{split}: cần {EXPECTED_COUNTS[split]} cặp, thấy {len(records)}')
    return records, image_dir, mask_dir

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f'Chưa Attach dataset {DATASET_HANDLE}; không thấy {DATA_ROOT}'
    )
train_records_all, train_image_dir, train_mask_dir = discover_records('train')
official_val_records_all, val_image_dir, val_mask_dir = discover_records('validation')
test_records_all, test_image_dir, test_mask_dir = discover_records('test')

split_ids = {
    'train': {record['image_id'] for record in train_records_all},
    'validation': {record['image_id'] for record in official_val_records_all},
    'test': {record['image_id'] for record in test_records_all},
}
assert split_ids['train'].isdisjoint(split_ids['validation'])
assert split_ids['train'].isdisjoint(split_ids['test'])
assert split_ids['validation'].isdisjoint(split_ids['test'])

def deterministic_subset(records, count, seed):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(records))[:min(count, len(records))]
    return [records[int(index)] for index in indices]

if FAST_DEV_RUN:
    train_records = deterministic_subset(train_records_all, 96, SEED)
    official_val_records = deterministic_subset(official_val_records_all, 24, SEED + 1)
    test_records = deterministic_subset(test_records_all, 24, SEED + 2)
else:
    train_records = train_records_all
    official_val_records = official_val_records_all
    test_records = test_records_all

generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(train_records), generator=generator).tolist()
fold_sizes = [len(train_records) // N_FOLDS] * N_FOLDS
for index in range(len(train_records) % N_FOLDS):
    fold_sizes[index] += 1
fold_indices, cursor = [], 0
for fold_size in fold_sizes:
    fold_indices.append(permutation[cursor:cursor + fold_size])
    cursor += fold_size
folds, all_indices = [], set(range(len(train_records)))
for holdout_indices in fold_indices:
    holdout_set = set(holdout_indices)
    train_indices = sorted(all_indices - holdout_set)
    train_subset = [train_records[index] for index in train_indices]
    holdout_subset = [train_records[index] for index in holdout_indices]
    assert {r['image_id'] for r in train_subset}.isdisjoint(
        r['image_id'] for r in holdout_subset
    )
    folds.append((train_subset, holdout_subset))

print('pairs:', {'train': len(train_records), 'validation': len(official_val_records),
                 'test': len(test_records)}, 'fold sizes:', fold_sizes)
print('train dirs:', train_image_dir, train_mask_dir, sep='\n  ')
print('validation dirs:', val_image_dir, val_mask_dir, sep='\n  ')
print('test dirs:', test_image_dir, test_mask_dir, sep='\n  ')

pairs: {'train': 2594, 'validation': 100, 'test': 1000} fold sizes: [865, 865, 864]
train dirs:
  /kaggle/input/datasets/tinlngnguyn/skin-lesion-segmentation-hus/ISIC2018_Task1-2_Training_Input/ISIC2018_Task1-2_Training_Input
  /kaggle/input/datasets/tinlngnguyn/skin-lesion-segmentation-hus/ISIC2018_Task1_Training_GroundTruth/ISIC2018_Task1_Training_GroundTruth
validation dirs:
  /kaggle/input/datasets/tinlngnguyn/skin-lesion-segmentation-hus/ISIC2018_Task1-2_Validation_Input/ISIC2018_Task1-2_Validation_Input
  /kaggle/input/datasets/tinlngnguyn/skin-lesion-segmentation-hus/ISIC2018_Task1_Validation_GroundTruth/ISIC2018_Task1_Validation_GroundTruth
test dirs:
  /kaggle/input/datasets/tinlngnguyn/skin-lesion-segmentation-hus/ISIC2018_Task1-2_Test_Input/ISIC2018_Task1-2_Test_Input
  /kaggle/input/datasets/tinlngnguyn/skin-lesion-segmentation-hus/ISIC2018_Task1_Test_GroundTruth/ISIC2018_Task1_Test_GroundTruth


In [7]:
IMAGENET_MEAN = [0.485, 0.456, 0.406] # chuẩn hóa ảnh theo ImageNet vì MobileNetV2 trong bài của bạn sử dụng trọng số pretrained (đã được huấn luyện trước trên ImageNet)
IMAGENET_STD = [0.229, 0.224, 0.225]
RAM_CACHE = {}

def gray_world(image, **kwargs):
    """Cân bằng màu Gray World."""
    del kwargs # kwargs cho phép hàm nhận thêm các tham số dạng từ điển.
    image_float = image.astype(np.float32) # Chuyển dữ liệu ảnh sang kiểu số thực float32.
    channel_means = image_float.mean(axis=(0, 1)) # Tính giá trị trung bình của từng kênh màu trên toàn bộ ảnh.
    gray_mean = channel_means.mean() #
    scale = gray_mean / np.maximum(channel_means, 1e-6)
    return np.clip(image_float * scale.reshape(1, 1, 3), 0, 255).astype(np.uint8)

def load_pil_uint8(path, mode):
    """ Đọc một file ảnh từ ổ đĩa, chuyển ảnh thành mảng NumPy kiểu uint8, sau đó trả về mảng ảnh đó. """
    with open(path, 'rb') as handle:
        with Image.open(handle) as source:
            array = np.asarray(source.convert(mode), dtype=np.uint8).copy()
        if (DROP_OS_PAGE_CACHE and hasattr(os, 'posix_fadvise')
                and hasattr(os, 'POSIX_FADV_DONTNEED')):
            try:
                os.posix_fadvise(handle.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
            except OSError:
                pass
    return array

def build_transform(size, train, seed, preprocessed=False):
    """
    Hàm build_transform() dùng để xây dựng quy trình biến đổi ảnh (image transformations) trước khi đưa ảnh vào mô hình MobileNetV2-UNet.
    Nó thực hiện những việc như:
    - Resize ảnh về kích thước mong muốn.
    - Cân bằng màu bằng Gray World.
    - Tạo ảnh biến đổi ngẫu nhiên để mô hình học tốt hơn.
    - Chuyển ảnh từ NumPy sang Tensor để PyTorch sử dụng.

    quá trình tăng cường dữ liệu (augentation)
    """
    steps = []
    if not preprocessed:
        steps.extend([
            A.Resize(size, size, interpolation=cv2.INTER_LINEAR,
                     mask_interpolation=cv2.INTER_NEAREST),
            A.Lambda(image=gray_world, p=1.0),
        ])
    if train:
        steps.extend([ 
            A.D4(p=1.0), # Phép 1: Xoay và lật ảnh
            A.Affine(rotate=(-44, 44), scale=(1.0, 1.05), # Phép 2: Biến đổi hình học
                     interpolation=cv2.INTER_LINEAR,
                     mask_interpolation=cv2.INTER_NEAREST, p=0.75),
            A.RandomBrightnessContrast(brightness_limit=0.2, # Phép 3: Thay đổi độ sáng và tương phản
                                       contrast_limit=0.2, p=0.75),
        ])
    steps.append(ToTensorV2()) # Chuyển đổi sang Tensor
    return A.Compose(steps, seed=int(seed)) # Kết hợp các bước thành một pipeline

class ISICDataset(Dataset):
    """
    Quản lý bộ dữ liệu: Hệ thống đọc và chuẩn bị dữ liệu ảnh ISIC để đưa vào mô hình MobileNetV2-UNet.
    """
    def __init__(self, records, size, train, seed=SEED):
        if not records:
            raise ValueError('Dataset rỗng')
        self.records = list(records)
        self.size = int(size)
        self.train = bool(train)
        cache = RAM_CACHE.get(self.size)
        indices = None if cache is None else [cache['id_to_index'].get(r['image_id'])
                                              for r in self.records]
        self.cache = cache if indices is not None and all(i is not None for i in indices) else None
        self.cache_indices = indices if self.cache is not None else None
        self.transform = build_transform(
            self.size, self.train, seed, preprocessed=self.cache is not None
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        if self.cache is not None:
            cache_index = self.cache_indices[index]
            image_np = self.cache['images'][cache_index].copy()
            mask_np = self.cache['masks'][cache_index].copy()
        else:
            image_np = load_pil_uint8(record['image_path'], 'RGB')
            mask_np = load_pil_uint8(record['mask_path'], 'L')
            if image_np.shape[:2] != mask_np.shape:
                raise ValueError(f"Image/mask lệch kích thước: {record['image_id']}")
            mask_np = (mask_np > 127).astype(np.uint8)
        transformed = self.transform(image=image_np, mask=mask_np)
        image, mask = transformed['image'], transformed['mask']
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        return image.contiguous(), (mask > 0).float().contiguous()

def seed_worker(worker_id):
    del worker_id
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    worker = get_worker_info()
    if worker is not None and hasattr(worker.dataset.transform, 'set_random_seed'):
        worker.dataset.transform.set_random_seed(worker_seed)

def make_loader(records, size, train, seed, batch_size=None, drop_last=False):
    if batch_size is None:
        batch_size = BATCH_SIZE_BY_SIZE[int(size)]
    kwargs = dict(
        dataset=ISICDataset(records, size, train, seed),
        batch_size=int(batch_size), shuffle=bool(train), drop_last=bool(drop_last),
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=False,
        worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(int(seed)),
    )
    if NUM_WORKERS > 0:
        kwargs['prefetch_factor'] = PREFETCH_FACTOR
    return DataLoader(**kwargs)

def build_ram_caches(records, sizes):
    records = list(records)
    sizes = tuple(dict.fromkeys(int(size) for size in sizes))
    image_ids = [record['image_id'] for record in records]
    if len(image_ids) != len(set(image_ids)):
        raise ValueError('Trùng image_id trong RAM cache')
    caches = {
        size: {
            'images': np.empty((len(records), size, size, 3), dtype=np.uint8),
            'masks': np.empty((len(records), size, size), dtype=np.uint8),
            'id_to_index': {image_id: index for index, image_id in enumerate(image_ids)},
        }
        for size in sizes
    }
    for index, record in enumerate(tqdm(records, desc=f'RAM cache {sizes}px')):
        image_np = load_pil_uint8(record['image_path'], 'RGB')
        mask_np = load_pil_uint8(record['mask_path'], 'L')
        if image_np.shape[:2] != mask_np.shape:
            raise ValueError(f"Image/mask lệch kích thước: {record['image_id']}")
        binary_mask = (mask_np > 127).astype(np.uint8)
        for size in sizes:
            resized = cv2.resize(image_np, (size, size), interpolation=cv2.INTER_LINEAR)
            caches[size]['images'][index] = gray_world(resized)
            caches[size]['masks'][index] = cv2.resize(
                binary_mask, (size, size), interpolation=cv2.INTER_NEAREST
            )
    return caches

if CACHE_RESIZED_IN_RAM:
    RAM_CACHE.update(build_ram_caches(train_records, SIZES))
    cache_gib = sum(v['images'].nbytes + v['masks'].nbytes
                    for v in RAM_CACHE.values()) / 2**30
    print(f'Train-only resized uint8 cache: {cache_gib:.2f} GiB | {memory_status()}')

clean_dataset = ISICDataset(train_records[:3], SIZES[0], train=False)
for index in range(len(clean_dataset)):
    x1, y1 = clean_dataset[index]
    x2, y2 = clean_dataset[index]
    assert torch.equal(x1, x2) and torch.equal(y1, y2)
    assert x1.dtype == torch.uint8 and y1.dtype == torch.float32
    assert set(torch.unique(y1).tolist()).issubset({0.0, 1.0})
    assert 0.0 < float(y1.mean()) < 1.0
del clean_dataset, x1, x2, y1, y2
print('Dataset, mask and clean-transform checks: OK')

RAM cache (128, 256)px:   0%|          | 0/2594 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# class DecoderBlock(nn.Module):
#     """Upsample -> ghép skip feature (nếu có) -> Conv -> BatchNorm -> ReLU."""
#     def __init__(self, in_channels, skip_channels, out_channels):
#         super().__init__()
#         self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
#         self.conv = nn.Sequential(
#             nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3,
#                       padding=1, bias=False),
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace=True),
#             nn.Conv2d(out_channels, out_channels, kernel_size=3,
#                       padding=1, bias=False),
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace=True),
#         )

#     def forward(self, x, skip=None):
#         x = self.upsample(x)
#         if skip is not None:
#             if x.shape[2:] != skip.shape[2:]:
#                 x = F.interpolate(x, size=skip.shape[2:],
#                                   mode='bilinear', align_corners=True)
#             x = torch.cat([x, skip], dim=1)
#         return self.conv(x)

class DecoderBlock(nn.Module):
    """
    ConvTranspose2d upsampling
    -> Skip projection
    -> Concatenate
    -> BatchNorm -> ReLU
    """

    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        # Upsample và giảm số kênh của feature đầu vào
        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels // 2,
            kernel_size=2,
            stride=2
        )

        # Chiếu skip feature về cùng số kênh
        self.skip_projection = nn.Conv2d(
            skip_channels,
            out_channels // 2,
            kernel_size=1
        )

        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x, skip):
        source_dtype = x.dtype

        # Upsampling
        if FP32_TRANSPOSED_CONV and x.is_cuda:
            with torch.autocast(device_type='cuda', enabled=False):
                x = self.up(x.float())
            x = x.to(dtype=source_dtype)
        else:
            x = self.up(x)

        # Điều chỉnh số kênh của skip feature
        skip = self.skip_projection(skip)

        # Điều chỉnh kích thước không gian nếu cần
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode='nearest'
            )

        # Ghép feature từ Decoder và Encoder
        x = torch.cat([x, skip], dim=1)

        # BatchNorm và ReLU
        return self.bn(F.relu(x))

# class MobileNetV2UNet(nn.Module):
#     """
#     MobileNetV2-UNet:
#     Input [B, 3, H, W] -> output raw logits [B, 1, H, W].
#     Kiến trúc lấy từ models/mobilenetv2_unet.py của nhánh branch_hoa.
#     """
#     def __init__(self, pretrained=True, out_channels=1):
#         super().__init__()
#         weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
#         try:
#             backbone = models.mobilenet_v2(weights=weights).features
#         except Exception as error:
#             raise RuntimeError(
#                 'Không tải được MobileNetV2 ImageNet weights; hãy bật Internet trong Kaggle.'
#             ) from error

#         # Encoder stages and skip connections from the repository model.
#         self.stage0 = backbone[0:2]    # /2, 16 channels
#         self.stage1 = backbone[2:4]    # /4, 24 channels
#         self.stage2 = backbone[4:7]    # /8, 32 channels
#         self.stage3 = backbone[7:14]   # /16, 96 channels
#         self.stage4 = backbone[14:18]  # /32, 320 channels

#         self.dec4 = DecoderBlock(320, 96, 256)
#         self.dec3 = DecoderBlock(256, 32, 128)
#         self.dec2 = DecoderBlock(128, 24, 64)
#         self.dec1 = DecoderBlock(64, 16, 32)
#         self.dec0 = DecoderBlock(32, 0, 16)
#         self.final_conv = nn.Conv2d(16, out_channels, kernel_size=1)

#     def encoder_modules(self):
#         """Cho phép freeze/unfreeze toàn bộ encoder theo hai giai đoạn."""
#         return [self.stage0, self.stage1, self.stage2, self.stage3, self.stage4]

#     def forward(self, x):
#         s0 = self.stage0(x)
#         s1 = self.stage1(s0)
#         s2 = self.stage2(s1)
#         s3 = self.stage3(s2)
#         s4 = self.stage4(s3)

#         x = self.dec4(s4, s3)
#         x = self.dec3(x, s2)
#         x = self.dec2(x, s1)
#         x = self.dec1(x, s0)
#         x = self.dec0(x)
#         return self.final_conv(x)  # raw logits, no sigmoid

class MobileNetV2UNet(nn.Module):
    """
    MobileNetV2-UNet:
    Input [B, 3, H, W] -> output raw logits [B, 1, H, W].
    """

    def __init__(self, pretrained=True, out_channels=1):
        super().__init__()

        # =========================
        # 1. ENCODER: MobileNetV2
        # =========================

        weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None

        try:
            backbone = models.mobilenet_v2(weights=weights).features
        except Exception as error:
            raise RuntimeError(
                'Không tải được MobileNetV2 ImageNet weights; '
                'hãy kiểm tra Internet trong Kaggle.'
            ) from error

        # Chia Encoder thành 5 stages
        self.stage0 = backbone[0:2]    # /2, 16 channels
        self.stage1 = backbone[2:4]    # /4, 24 channels
        self.stage2 = backbone[4:7]    # /8, 32 channels
        self.stage3 = backbone[7:14]   # /16, 96 channels
        self.stage4 = backbone[14:18]  # /32, 320 channels

        # =========================
        # 2. DECODER
        # =========================

        self.dec4 = DecoderBlock(320, 96, 256)
        self.dec3 = DecoderBlock(256, 32, 128)
        self.dec2 = DecoderBlock(128, 24, 64)
        self.dec1 = DecoderBlock(64, 16, 32)
        self.dec0 = DecoderBlock(32, 0, 16)

        # =========================
        # 3. OUTPUT
        # =========================

        self.final_conv = nn.Conv2d(
            16,
            out_channels,
            kernel_size=1
        )

    def encoder_modules(self):
        """Trả về các stage của Encoder để freeze/unfreeze."""
        return [
            self.stage0,
            self.stage1,
            self.stage2,
            self.stage3,
            self.stage4
        ]

    def forward(self, x):

        # Encoder
        s0 = self.stage0(x)
        s1 = self.stage1(s0)
        s2 = self.stage2(s1)
        s3 = self.stage3(s2)
        s4 = self.stage4(s3)

        # Decoder + Skip Connections
        x = self.dec4(s4, s3)
        x = self.dec3(x, s2)
        x = self.dec2(x, s1)
        x = self.dec1(x, s0)
        x = self.dec0(x)

        # Output logits
        return self.final_conv(x)
        


# ============================================================
# 1. UNWRAP MODEL
# ============================================================

def unwrap_model(model):
    """
    Lấy mô hình gốc.

    Hiện tại notebook không sử dụng DataParallel hoặc DDP,
    nên chỉ cần trả về chính model.
    """
    return model


# ============================================================
# 2. TO RUNTIME
# ============================================================

def to_runtime(model):
    """
    Chuyển mô hình lên thiết bị tính toán (GPU hoặc CPU).

    DEVICE:
        - 'cuda:0': sử dụng GPU đầu tiên.
        - 'cpu': sử dụng CPU.

    USE_CHANNELS_LAST:
        Nếu bật, chuyển cách sắp xếp bộ nhớ tensor sang
        channels_last, có thể giúp tăng tốc một số phép tính
        trên GPU.
    """

    # Chuyển toàn bộ mô hình lên thiết bị được chỉ định
    model = model.to(DEVICE)

    # Tùy chọn thay đổi memory format
    if USE_CHANNELS_LAST:
        model = model.to(memory_format=torch.channels_last)

    return model


# ============================================================
# 3. FREEZE ENCODER
# ============================================================

def freeze_first_layer_group(model):
    """
    Đóng băng Encoder trong giai đoạn huấn luyện đầu tiên.

    Mục tiêu:
        - Giữ nguyên phần lớn trọng số Encoder đã pretrained.
        - Chỉ huấn luyện Decoder và các phần còn lại.
        - Riêng tham số affine của BatchNorm trong Encoder
          vẫn được phép cập nhật.

    requires_grad:
        True  -> tham số được tính gradient và có thể cập nhật.
        False -> tham số không được cập nhật bằng gradient.
    """

    # Lấy mô hình gốc
    base = unwrap_model(model)

    # Ban đầu cho phép tất cả tham số được cập nhật
    for parameter in base.parameters():
        parameter.requires_grad = True

    # Duyệt qua các stage thuộc Encoder
    for module in base.encoder_modules():

        # Duyệt qua từng lớp bên trong stage
        for child in module.modules():

            # Chỉ lấy tham số trực tiếp của từng lớp
            for parameter in child.parameters(recurse=False):

                # Chỉ giữ tham số BatchNorm được cập nhật
                # Các lớp khác trong Encoder sẽ bị đóng băng
                parameter.requires_grad = isinstance(
                    child, nn.BatchNorm2d
                )


# ============================================================
# 4. UNFREEZE ENCODER, FREEZE BATCHNORM
# ============================================================

def unfreeze_with_frozen_bn(model):
    """
    Mở khóa Encoder trong giai đoạn fine-tuning.

    Mục tiêu:
        - Cho phép các trọng số Encoder được cập nhật.
        - Giữ cố định các tham số của BatchNorm.

    Lưu ý:
        Hàm này đóng băng tham số BatchNorm thông qua
        requires_grad=False.
        Trạng thái train/eval của BatchNorm được xử lý
        riêng trong set_train_mode().
    """

    # Cho phép tất cả tham số được cập nhật
    for parameter in model.parameters():
        parameter.requires_grad = True

    # Đóng băng tham số của tất cả lớp BatchNorm
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):

            for parameter in module.parameters():
                parameter.requires_grad = False


# ============================================================
# 5. SET TRAIN MODE
# ============================================================

def set_train_mode(model, freeze_bn):
    """
    Thiết lập chế độ hoạt động của mô hình khi huấn luyện.

    model.train():
        Đưa mô hình về chế độ training.

    freeze_bn=True:
        Chuyển các lớp BatchNorm sang chế độ eval,
        không cập nhật running_mean và running_var.

    Lưu ý:
        eval() không có nghĩa là ngừng tính gradient.
        Nó chỉ thay đổi cách các lớp như BatchNorm hoạt động.
    """

    # Đưa toàn bộ mô hình về chế độ training
    model.train()

    # Nếu cần đóng băng hoạt động thống kê của BatchNorm
    if freeze_bn:

        for module in model.modules():

            if isinstance(module, nn.BatchNorm2d):

                # BatchNorm sử dụng running statistics
                # thay vì cập nhật theo batch hiện tại
                module.eval()


# ============================================================
# 6. COPY MODEL STATE TO CPU
# ============================================================

def cpu_state_dict(model):
    """
    Sao chép trạng thái hiện tại của mô hình sang CPU.

    state_dict() bao gồm:
        - Trọng số các lớp Convolution.
        - Tham số BatchNorm.
        - running_mean và running_var.
        - Các buffer khác của mô hình.

    Mục đích:
        Lưu một bản sao độc lập trên CPU để quản lý
        checkpoint mà không cần giữ bản sao trên GPU.
    """

    return {
        key: value.detach().cpu().clone()
        for key, value in unwrap_model(model).state_dict().items()
    }


# ============================================================
# 7. REFRESH CPU STATE DICT
# ============================================================

@torch.no_grad()
def refresh_cpu_state_dict(model, destination):
    """
    Cập nhật bản sao state_dict trên CPU bằng trạng thái
    mới nhất của mô hình.

    Ví dụ:
        - destination đang lưu trọng số ở epoch trước.
        - Sau khi train, trọng số mô hình đã thay đổi.
        - Hàm này sao chép trọng số mới vào destination.

    @torch.no_grad():
        Không cần tính gradient khi chỉ sao chép trọng số.
    """

    # Lấy trạng thái hiện tại của mô hình
    source = unwrap_model(model).state_dict()

    # Kiểm tra cấu trúc state_dict có còn giống nhau không
    if tuple(source) != tuple(destination):
        raise RuntimeError(
            'state_dict keys changed during training'
        )

    # Sao chép từng tensor từ mô hình sang bản lưu CPU
    for key, value in source.items():

        destination[key].copy_(
            value.detach(),
            non_blocking=False
        )


# ============================================================
# 8. LOAD MODEL STATE
# ============================================================

def load_model_state(model, state):
    """
    Nạp trọng số đã lưu vào mô hình.

    state:
        State dictionary được lấy từ checkpoint.

    Xử lý tiền tố 'module.':
        Một số checkpoint được lưu từ DataParallel/DDP
        có key dạng 'module.xxx'.

        Nếu có, loại bỏ tiền tố này trước khi load.
    """

    # Loại bỏ tiền tố module. nếu toàn bộ key đều có tiền tố đó
    while state and all(
        key.startswith('module.') for key in state
    ):

        state = {
            key.removeprefix('module.'): value
            for key, value in state.items()
        }

    # Nạp trọng số vào mô hình
    unwrap_model(model).load_state_dict(state)


# ============================================================
# 9. MODEL SHAPE CHECK
# ============================================================

# Tạo một mô hình MobileNetV2-UNet để kiểm tra
# pretrained=False: không cần tải ImageNet weights
# eval(): chuyển sang chế độ đánh giá
shape_model = MobileNetV2UNet(pretrained=False).eval()


# Tạo dữ liệu giả để kiểm tra luồng truyền dữ liệu
# 2       : batch size = 2 ảnh
# 3       : 3 kênh RGB
# 128,128 : chiều cao và chiều rộng ảnh
with torch.inference_mode():

    shape_output = shape_model(
        torch.randn(2, 3, 128, 128)
    )


# Kiểm tra kích thước đầu ra:
# [Batch, Channels, Height, Width]
# Mong đợi: [2, 1, 128, 128]
assert shape_output.shape == (2, 1, 128, 128), shape_output.shape


# Kiểm tra mô hình không chứa module first_layer_group.
# Đây là kiểm tra cấu trúc state_dict theo thiết kế notebook.
assert not any(
    key.startswith('first_layer_group.')
    for key in shape_model.state_dict()
)


# Đếm tổng số tham số của mô hình
# numel() = số phần tử trong một tensor
# Chia 1e6 để hiển thị theo đơn vị triệu tham số (M)
print(
    f'Parameters: '
    f'{sum(p.numel() for p in shape_model.parameters()) / 1e6:.3f}M'
)


# Xóa mô hình kiểm tra và dữ liệu giả
# Giúp giải phóng bộ nhớ không còn cần thiết
del shape_model, shape_output



In [ ]:
c
# ============================================================
# 1. LOSS FUNCTION VÀ CÁC THAM SỐ CHUẨN HÓA ẢNH
# ============================================================

# Hàm mất mát dùng trong quá trình huấn luyện.
# BCEWithLogitsLoss nhận đầu vào là raw logits (chưa qua Sigmoid)
# và mask thực tế (ground truth).
# Mục tiêu: đo mức độ sai lệch giữa dự đoán và mask thực tế.
criterion = nn.BCEWithLogitsLoss()


# Xác định kiểu dữ liệu của ảnh đầu vào:
# - float16: nếu bật Automatic Mixed Precision (AMP).
# - float32: nếu không bật AMP.
# float16 giúp giảm bộ nhớ GPU và có thể tăng tốc tính toán.
INPUT_DTYPE = torch.float16 if USE_AMP else torch.float32


# Tạo tensor Mean của ImageNet trên thiết bị DEVICE.
# IMAGENET_MEAN = [0.485, 0.456, 0.406] tương ứng RGB.
# .view(1, 3, 1, 1): định dạng để áp dụng cho batch ảnh
# có kích thước [Batch, Channels, Height, Width].
GPU_MEAN = torch.tensor(
    IMAGENET_MEAN,
    device=DEVICE,
    dtype=INPUT_DTYPE
).view(1, 3, 1, 1)


# Tạo tensor Std của ImageNet trên thiết bị DEVICE.
# Dùng để chuẩn hóa ảnh theo thống kê ImageNet,
# phù hợp với trọng số pretrained của MobileNetV2.
GPU_STD = torch.tensor(
    IMAGENET_STD,
    device=DEVICE,
    dtype=INPUT_DTYPE
).view(1, 3, 1, 1)


# ============================================================
# 2. CHUẨN BỊ ẢNH ĐẦU VÀO
# ============================================================

def prepare_inputs(inputs):
    """
    Chuẩn bị ảnh trước khi đưa vào mô hình.

    Đầu vào:
        inputs: batch ảnh RGB, thường có giá trị pixel 0-255.

    Các bước:
        1. Chuyển ảnh lên GPU/CPU.
        2. Chuyển kiểu dữ liệu.
        3. Chia 255 để đưa pixel về khoảng 0-1.
        4. Chuẩn hóa theo Mean và Std của ImageNet.
        5. Sắp xếp bộ nhớ tensor phù hợp với mô hình.
    """

    # Chuyển batch ảnh lên thiết bị tính toán.
    # non_blocking=True: cho phép truyền dữ liệu bất đồng bộ
    # trong những điều kiện bộ nhớ phù hợp.
    inputs = inputs.to(
        DEVICE,
        non_blocking=True
    ).to(dtype=INPUT_DTYPE)

    # Chuẩn hóa ảnh theo công thức:
    # (pixel / 255 - ImageNet Mean) / ImageNet Std
    #
    # Các phép toán có dấu "_" là phép toán thực hiện trực tiếp
    # trên tensor hiện tại, không tạo tensor mới cho từng bước.
    inputs.div_(255.0).sub_(GPU_MEAN).div_(GPU_STD)

    # Nếu bật CHANNELS_LAST thì chuyển sang memory format này.
    # Nếu không, sử dụng định dạng tensor liên tục thông thường.
    if USE_CHANNELS_LAST:
        return inputs.contiguous(
            memory_format=torch.channels_last
        )

    return inputs.contiguous()


# ============================================================
# 3. TÍNH DICE CHO TỪNG ẢNH
# ============================================================

def per_image_dice(probability, target, threshold=PRED_THRESHOLD):
    """
    Tính Dice riêng cho từng ảnh trong batch.

    probability:
        Xác suất dự đoán từ mô hình sau khi qua Sigmoid.

    target:
        Mask thực tế (ground truth), giá trị 0 hoặc 1.

    threshold:
        Ngưỡng chuyển xác suất thành mask nhị phân.
        Mặc định là PRED_THRESHOLD = 0.5.

    Kết quả:
        Một giá trị Dice cho mỗi ảnh trong batch.
    """

    # Chuyển xác suất thành mask nhị phân:
    # - Lớn hơn threshold -> 1 (vùng tổn thương).
    # - Nhỏ hơn hoặc bằng threshold -> 0 (nền).
    prediction = (probability > threshold).float()

    # Tính toán trên 3 chiều:
    # Channel, Height, Width.
    # Giữ lại chiều Batch để tính Dice riêng từng ảnh.
    dims = (1, 2, 3)

    # Tính số pixel thuộc vùng giao nhau giữa:
    # - prediction: mask mô hình dự đoán.
    # - target: mask thực tế.
    intersection = (prediction * target).sum(dims)

    # Tổng số pixel vùng dự đoán + tổng số pixel vùng thực tế.
    denominator = prediction.sum(dims) + target.sum(dims)

    # Công thức Dice:
    # Dice = 2 * Intersection / (Prediction + Ground Truth)
    #
    # Nếu cả hai mask đều rỗng thì quy ước Dice = 1.
    # clamp(min=1) tránh phép chia cho 0.
    return torch.where(
        denominator > 0,
        2 * intersection / denominator.clamp(min=1),
        torch.ones_like(denominator)
    )


# ============================================================
# 4. ĐÁNH GIÁ DICE TRÊN MỘT TẬP DỮ LIỆU
# ============================================================

@torch.inference_mode()
def evaluate_dice(model, loader):
    """
    Đánh giá Dice trung bình của một mô hình trên DataLoader.

    model:
        Mô hình segmentation cần đánh giá.

    loader:
        DataLoader chứa ảnh và mask thực tế.

    Kết quả:
        Dice trung bình trên toàn bộ ảnh trong loader.

    inference_mode():
        Không lưu thông tin gradient vì đây là giai đoạn đánh giá.
        Giúp tiết kiệm bộ nhớ và tăng hiệu quả tính toán.
    """

    # Chuyển mô hình sang chế độ đánh giá.
    # BatchNorm sử dụng running statistics thay vì cập nhật
    # theo batch hiện tại.
    model.eval()

    # Biến lưu tổng Dice của tất cả ảnh.
    dice_sum = torch.zeros((), device=DEVICE)

    # Đếm tổng số ảnh đã đánh giá.
    sample_count = 0

    # Duyệt qua từng batch ảnh và mask thực tế.
    for inputs, targets in loader:

        # Chuẩn hóa ảnh và chuyển lên thiết bị tính toán.
        inputs = prepare_inputs(inputs)

        # Chuyển mask thực tế lên cùng thiết bị với mô hình.
        targets = targets.to(
            DEVICE,
            non_blocking=True
        )

        # Automatic Mixed Precision:
        # Nếu USE_AMP=True, sử dụng float16 cho các phép tính
        # phù hợp để tiết kiệm bộ nhớ và tăng tốc.
        with torch.autocast(
            device_type='cuda',
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            # Đưa ảnh qua mô hình để lấy raw logits.
            # .float() chuyển logits về float32.
            # Sigmoid chuyển logits thành xác suất trong khoảng 0-1.
            probability = torch.sigmoid(
                model(inputs).float()
            )

        # Kiểm tra xác suất có chứa NaN hoặc Inf không.
        # Nếu có, dừng đánh giá và báo lỗi.
        if not bool(torch.isfinite(probability).all()):
            raise FloatingPointError(
                'Model sinh xác suất NaN/Inf'
            )

        # Tính Dice cho từng ảnh trong batch
        # rồi cộng vào tổng Dice.
        dice_sum += per_image_dice(
            probability,
            targets
        ).sum()

        # Cộng số ảnh trong batch vào tổng số ảnh.
        sample_count += targets.shape[0]

    # Kiểm tra DataLoader có chứa dữ liệu hay không.
    if sample_count == 0:
        raise RuntimeError('Holdout loader rỗng')

    # Trả về Dice trung bình của toàn bộ tập dữ liệu.
    # .cpu(): chuyển kết quả về CPU.
    # float(): chuyển tensor thành số thực Python.
    return float((dice_sum / sample_count).cpu())


# ============================================================
# 5. TÍNH CÁC CHỈ SỐ: IoU, DICE, SENSITIVITY
# ============================================================

def per_image_scores(probability, target, threshold=PRED_THRESHOLD):
    """
    Tính các chỉ số đánh giá riêng cho từng ảnh.

    Các chỉ số:
        1. Jaccard / IoU: mức độ trùng khớp giữa hai vùng.
        2. Dice: mức độ tương đồng giữa hai mask.
        3. Sensitivity / Recall: khả năng phát hiện vùng tổn thương.

    Trả về:
        jaccard, dice, sensitivity.
    """

    # Chuyển xác suất thành mask nhị phân theo threshold.
    prediction = (probability > threshold).float()

    # Tính toán riêng trên từng ảnh.
    dims = (1, 2, 3)

    # Số pixel thuộc vùng giao nhau giữa dự đoán và thực tế.
    intersection = (prediction * target).sum(dims)

    # Tổng số pixel thuộc vùng dự đoán.
    prediction_sum = prediction.sum(dims)

    # Tổng số pixel thuộc vùng thực tế.
    target_sum = target.sum(dims)

    # Union = Prediction + Ground Truth - Intersection.
    # Tức là tổng vùng hợp nhất của hai mask.
    union = prediction_sum + target_sum - intersection

    # --------------------------------------------------------
    # 5.1. JACCARD / IoU
    # --------------------------------------------------------

    # IoU = Intersection / Union.
    # Nếu cả hai mask đều rỗng thì quy ước IoU = 1.
    jaccard = torch.where(
        union > 0,
        intersection / union.clamp(min=1),
        torch.ones_like(union)
    )

    # --------------------------------------------------------
    # 5.2. DICE
    # --------------------------------------------------------

    # Dice = 2 * Intersection / (Prediction + Ground Truth).
    # Nếu cả hai mask đều rỗng thì quy ước Dice = 1.
    dice = torch.where(
        prediction_sum + target_sum > 0,
        2 * intersection /
        (prediction_sum + target_sum).clamp(min=1),
        torch.ones_like(union)
    )

    # --------------------------------------------------------
    # 5.3. SENSITIVITY / RECALL
    # --------------------------------------------------------

    # Sensitivity = TP / (TP + FN).
    # Đo tỷ lệ pixel tổn thương thực tế được mô hình phát hiện.
    #
    # Nếu target rỗng thì code quy ước Sensitivity = 1.
    sensitivity = torch.where(
        target_sum > 0,
        intersection / target_sum.clamp(min=1),
        torch.ones_like(target_sum)
    )

    # Trả về 3 chỉ số cho từng ảnh.
    return jaccard, dice, sensitivity


# ============================================================
# 6. LẤY ĐƯỜNG BIÊN CỦA MASK
# ============================================================

def mask_surface(mask):
    """
    Trích xuất đường biên của mask nhị phân.

    mask:
        Mask Boolean, trong đó:
        - True: vùng tổn thương.
        - False: nền.

    Các bước:
        1. Erosion: làm co vùng mask.
        2. XOR: lấy phần khác nhau giữa mask ban đầu
           và mask sau khi co.

    Kết quả:
        Mask chỉ chứa các pixel thuộc đường biên.
    """

    # Làm co vùng mask.
    # border_value=0: xem phần nằm ngoài ảnh là nền.
    eroded = ndimage.binary_erosion(
        mask,
        border_value=0
    )

    # Lấy phần khác nhau giữa mask gốc và mask đã co.
    # Phần còn lại chính là đường biên.
    return np.logical_xor(mask, eroded)


# ============================================================
# 7. TÍNH HD95 CHO MỘT ẢNH
# ============================================================

def hd95_single(prediction, target):
    """
    Tính HD95 (95th Percentile Hausdorff Distance)
    giữa mask dự đoán và mask thực tế của một ảnh.

    HD95 đo khoảng cách giữa hai đường biên.

    Ý nghĩa:
        - HD95 càng nhỏ: đường biên dự đoán càng gần thực tế.
        - HD95 = 0: hai đường biên trùng nhau.

    Đơn vị hiện tại: pixel.
    """

    # Chuyển hai mask về kiểu Boolean.
    prediction = prediction.astype(bool)
    target = target.astype(bool)

    # Lấy chiều cao và chiều rộng của mask.
    height, width = target.shape

    # Trường hợp cả hai mask đều rỗng:
    # Không có vùng phân đoạn ở cả dự đoán và thực tế.
    # Quy ước HD95 = 0.
    if not prediction.any() and not target.any():
        return 0.0

    # Trường hợp chỉ một trong hai mask rỗng:
    # Trả về đường chéo ảnh làm giá trị phạt.
    if not prediction.any() or not target.any():
        return float(math.hypot(height, width))

    # Trích xuất đường biên của mask dự đoán.
    prediction_surface = mask_surface(prediction)

    # Trích xuất đường biên của mask thực tế.
    target_surface = mask_surface(target)

    # Tính khoảng cách từ mỗi pixel đến đường biên thực tế.
    # ~target_surface: các pixel không thuộc đường biên thực tế.
    distance_to_target = ndimage.distance_transform_edt(
        ~target_surface
    )

    # Tính khoảng cách từ mỗi pixel đến đường biên dự đoán.
    distance_to_prediction = ndimage.distance_transform_edt(
        ~prediction_surface
    )

    # Lấy khoảng cách:
    # - Từ đường biên dự đoán đến đường biên thực tế.
    # - Từ đường biên thực tế đến đường biên dự đoán.
    #
    # Sau đó ghép hai tập khoảng cách lại.
    distances = np.concatenate([
        distance_to_target[prediction_surface],
        distance_to_prediction[target_surface]
    ])

    # Lấy phân vị thứ 95 của tập khoảng cách.
    # Đây là giá trị HD95.
    # Nếu không có khoảng cách thì trả về 0.
    return (
        float(np.percentile(distances, 95))
        if distances.size else 0.0
    )


# ============================================================
# 8. ENSEMBLE: KẾT HỢP DỰ ĐOÁN TỪ NHIỀU MÔ HÌNH
# ============================================================

@torch.inference_mode()
def ensemble_probability(models_list, inputs):
    """
    Kết hợp xác suất dự đoán từ nhiều mô hình bằng cách lấy trung bình.

    models_list:
        Danh sách các mô hình cần kết hợp.

    inputs:
        Batch ảnh đã được chuẩn hóa và chuyển lên thiết bị.

    Kết quả:
        Xác suất dự đoán trung bình của các mô hình.

    Ví dụ:
        Model 1 dự đoán: 0.8
        Model 2 dự đoán: 0.7
        Model 3 dự đoán: 0.9

        Ensemble = (0.8 + 0.7 + 0.9) / 3 = 0.8
    """

    # Tạo tensor để lưu tổng xác suất dự đoán.
    # Kích thước:
    # [Batch, 1, Height, Width].
    #
    # float32 được dùng để cộng xác suất ổn định hơn.
    total = torch.zeros(
        (
            inputs.shape[0],
            1,
            inputs.shape[2],
            inputs.shape[3]
        ),
        device=inputs.device,
        dtype=torch.float32
    )

    # Duyệt qua từng mô hình trong danh sách.
    for model in models_list:

        # Chuyển mô hình sang chế độ đánh giá.
        model.eval()

        # Dự đoán bằng mô hình hiện tại.
        with torch.autocast(
            device_type='cuda',
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            # Lấy raw logits -> float32 -> Sigmoid -> xác suất.
            probability = torch.sigmoid(
                model(inputs).float()
            )

        # Kiểm tra xác suất có chứa NaN hoặc Inf không.
        if not bool(torch.isfinite(probability).all()):
            raise FloatingPointError(
                'Model sinh xác suất NaN/Inf khi evaluation'
            )

        # Cộng xác suất của mô hình hiện tại vào tổng.
        total += probability

    # Chia cho số lượng mô hình để lấy xác suất trung bình.
    return total / len(models_list)


# ============================================================
# 9. ĐÁNH GIÁ ĐẦY ĐỦ CÁC CHỈ SỐ
# ============================================================

@torch.inference_mode()
def evaluate_full(models_list, loader, include_hd95=True):
    """
    Đánh giá mô hình hoặc Ensemble trên toàn bộ DataLoader.

    models_list:
        Danh sách mô hình cần đánh giá.
        Có thể chứa một mô hình hoặc nhiều mô hình.

    loader:
        DataLoader chứa ảnh và mask thực tế.

    include_hd95:
        True  -> tính thêm HD95.
        False -> bỏ qua HD95 để đánh giá nhanh hơn.

    Các chỉ số trả về:
        - jaccard: IoU trung bình.
        - threshold_jaccard: IoU chỉ tính khi đạt ngưỡng.
        - dice: Dice trung bình.
        - sensitivity: Sensitivity trung bình.
        - hd95: HD95 trung bình (nếu include_hd95=True).
    """

    # Khởi tạo biến lưu tổng giá trị của từng chỉ số.
    sums = {
        'jaccard': 0.0,
        'threshold_jaccard': 0.0,
        'dice': 0.0,
        'sensitivity': 0.0
    }

    # Biến lưu tổng HD95 và tổng số ảnh đã đánh giá.
    hd95_sum = 0.0
    sample_count = 0

    # Duyệt qua từng batch ảnh và mask thực tế.
    for inputs, targets in loader:

        # Chuẩn hóa ảnh và chuyển lên thiết bị tính toán.
        inputs = prepare_inputs(inputs)

        # Chuyển mask thực tế lên thiết bị tính toán.
        targets_device = targets.to(
            DEVICE,
            non_blocking=True
        )

        # Lấy xác suất dự đoán trung bình từ các mô hình.
        probability = ensemble_probability(
            models_list,
            inputs
        )

        # Tính IoU, Dice và Sensitivity cho từng ảnh.
        jaccard, dice, sensitivity = per_image_scores(
            probability,
            targets_device
        )

        # Cộng tổng IoU của batch hiện tại.
        sums['jaccard'] += float(jaccard.sum())

        # Chỉ cộng IoU của những ảnh đạt ngưỡng JACCARD_CUT.
        # Ví dụ JACCARD_CUT = 0.65:
        # - IoU >= 0.65: giữ nguyên giá trị IoU.
        # - IoU < 0.65: tính bằng 0.
        #
        # Sau đó vẫn chia cho toàn bộ số ảnh.
        sums['threshold_jaccard'] += float(
            (jaccard * (jaccard >= JACCARD_CUT)).sum()
        )

        # Cộng tổng Dice của batch.
        sums['dice'] += float(dice.sum())

        # Cộng tổng Sensitivity của batch.
        sums['sensitivity'] += float(sensitivity.sum())

        # Cập nhật số lượng ảnh đã đánh giá.
        sample_count += targets.shape[0]

        # ----------------------------------------------------
        # TÍNH HD95 NẾU ĐƯỢC YÊU CẦU
        # ----------------------------------------------------

        if include_hd95:

            # Chuyển xác suất thành mask nhị phân.
            # .cpu().numpy(): chuyển tensor sang NumPy để
            # sử dụng các hàm xử lý ảnh của SciPy.
            prediction_np = (
                probability > PRED_THRESHOLD
            ).cpu().numpy()[:, 0]

            # Chuyển mask thực tế về NumPy và nhị phân hóa.
            # targets ban đầu nằm trên CPU.
            target_np = (
                targets.numpy()[:, 0] > 0.5
            )

            # Tính HD95 cho từng cặp mask dự đoán và thực tế.
            # Sau đó cộng vào tổng HD95.
            hd95_sum += sum(
                hd95_single(prediction, target)
                for prediction, target in zip(
                    prediction_np,
                    target_np
                )
            )

    # Kiểm tra DataLoader có dữ liệu hay không.
    if sample_count == 0:
        raise RuntimeError('Evaluation loader rỗng')

    # Tính trung bình các chỉ số bằng cách chia tổng
    # cho tổng số ảnh đã đánh giá.
    metrics = {
        key: value / sample_count
        for key, value in sums.items()
    }

    # Nếu có tính HD95 thì bổ sung HD95 trung bình.
    if include_hd95:
        metrics['hd95'] = hd95_sum / sample_count

    # Trả về dictionary chứa các chỉ số đánh giá.
    return metrics


# ============================================================
# 10. KIỂM TRA HÀM TÍNH DICE
# ============================================================

# Tạo hai mask thực tế giả.
# Kích thước:
# [2, 1, 8, 8]
# 2 ảnh, 1 kênh, chiều cao 8, chiều rộng 8.
metric_target = torch.zeros(
    2, 1, 8, 8,
    device=DEVICE
)

# Ảnh thứ nhất:
# Đặt 4 hàng đầu tiên thành vùng tổn thương (giá trị 1).
metric_target[0, :, :4] = 1

# Ảnh thứ hai:
# Đặt 4 cột đầu tiên thành vùng tổn thương.
metric_target[1, :, :, :4] = 1


# Tạo mask dự đoán giả bằng cách sao chép mask thực tế.
metric_probability = metric_target.clone()

# Với ảnh thứ hai, xóa toàn bộ dự đoán ban đầu.
metric_probability[1] = 0

# Sau đó chỉ dự đoán đúng 2 cột đầu tiên.
metric_probability[1, :, :, :2] = 1


# Kiểm tra kết quả Dice:
#
# Ảnh 1:
# Prediction trùng hoàn toàn Ground Truth -> Dice = 1.
#
# Ảnh 2:
# Prediction có 16 pixel đúng.
# Ground Truth có 32 pixel.
# Prediction cũng có 16 pixel.
#
# Dice = 2 * 16 / (16 + 32) = 2/3.
#
# Kết quả mong đợi: [1.0, 2/3].
assert torch.allclose(
    per_image_dice(
        metric_probability,
        metric_target
    ),
    torch.tensor(
        [1.0, 2 / 3],
        device=DEVICE
    ),
    atol=1e-6
)

# Xóa các tensor kiểm tra để giải phóng bộ nhớ.
del metric_target, metric_probability

# Thông báo kiểm tra Dice thành công.
print('Dice metric check: OK')


In [ ]:
    if saved is None:
        selected_lr, lrf_lrs, lrf_losses = lr_find(
            model, train_subset, size, stage_seed, freeze_bn
        )
        initial_dice = evaluate_dice(model, holdout_loader)
        best_score, best_epoch = initial_dice, 0
        history = []
        best_state = cpu_state_dict(model)
        best_meta = {
            'epoch': best_epoch, 'dice': best_score, 'selected_lr': selected_lr,
            'contract': contract,
        }
        save_best_state(best_state, best_path, best_meta)
        start_epoch = 0
    else:
        selected_lr = float(saved['selected_lr'])
        lrf_lrs, lrf_losses = saved['lrf_lrs'], saved['lrf_losses']
        best_score = float(saved['best_score'])
        best_epoch = int(saved['best_epoch'])
        history = list(saved['history'])
        best_payload = trusted_torch_load(best_path, map_location='cpu')
        best_meta = best_payload.get('meta', {})
        if best_meta.get('contract') != contract:
            raise RuntimeError(f'Best checkpoint không khớp: {best_path}')
        if int(best_meta['epoch']) >= best_epoch:
            best_epoch = int(best_meta['epoch'])
            best_score = float(best_meta['dice'])
        best_state = best_payload['model_state']
        del best_payload
        start_epoch = int(saved['next_epoch'])
        load_model_state(model, saved['model_state'])
        print(f'[resume] fold={fold_number} {size}px {stage_name}: epoch {start_epoch + 1}')

    steps_per_epoch = math.ceil(len(train_subset) / BATCH_SIZE_BY_SIZE[size])
    optimizer = optimizer_for(model, selected_lr)
    scaler = make_scaler()
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, stlr_lambda(epochs * steps_per_epoch)
    )
    if saved is not None:
        optimizer.load_state_dict(saved['optimizer_state'])
        scheduler.load_state_dict(saved['scheduler_state'])
        scaler.load_state_dict(saved['scaler_state'])
        saved = None
        gc.collect()

    for epoch in range(start_epoch, epochs):
        epoch_started = time.perf_counter()
        train_loader = make_loader(train_subset, size, train=True,
                                   seed=stage_seed + epoch, drop_last=False)
        set_train_mode(model, freeze_bn)
        loss_sum = torch.zeros((), device=DEVICE)
        samples_seen = 0
        for inputs, targets in train_loader:
            loss_sum += train_batch(model, optimizer, scaler, inputs, targets)
            samples_seen += inputs.shape[0]
            # Cập nhật Learning Rate sau mỗi batch, theo lịch STLR.
            scheduler.step()
        del inputs, targets
        train_loss = float((loss_sum / steps_per_epoch).cpu())
        del train_loader
        holdout_dice = evaluate_dice(model, holdout_loader)
        row = {
            'fold': fold_number, 'size': size, 'stage': stage_name,
            'epoch': epoch + 1, 'train_loss': train_loss,
            'holdout_dice': holdout_dice, 'lr': optimizer.param_groups[0]['lr'],
        }
        history.append(row)
        # Chỉ thay checkpoint tốt nhất khi Holdout Dice được cải thiện.
        if holdout_dice > best_score:
            best_score, best_epoch = holdout_dice, epoch + 1
            refresh_cpu_state_dict(model, best_state)
        if (epoch + 1) % SAVE_LAST_EVERY == 0 or epoch + 1 == epochs:
            best_meta = {
                'epoch': best_epoch, 'dice': best_score, 'selected_lr': selected_lr,
                'contract': contract,
            }
            save_best_state(best_state, best_path, best_meta)
            # Gói trạng thái hiện tại để nếu phiên Kaggle bị ngắt có thể resume.
            progress = {
                'complete': False, 'contract': contract, 'next_epoch': epoch + 1,
                'selected_lr': selected_lr, 'lrf_lrs': lrf_lrs,
                'lrf_losses': lrf_losses, 'best_score': best_score,
                'best_epoch': best_epoch, 'history': history,
                'model_state': unwrap_model(model).state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'scaler_state': scaler.state_dict(),
            }
            atomic_torch_save(progress, last_path)
            del progress
        seconds = time.perf_counter() - epoch_started
        print(
            f'[fold {fold_number} | {size}px | {stage_name}] '
            f'epoch {epoch + 1:02d}/{epochs} loss={train_loss:.4f} '
            f'holdout_dice={holdout_dice:.4f} best={best_score:.4f} '
            f'time={seconds:.1f}s img/s={samples_seen / seconds:.1f} {memory_status()}'
        )

    # Khi kết thúc, nạp lại epoch có Holdout Dice tốt nhất, không mặc định lấy epoch cuối.
    load_model_state(model, best_state)
    result = {
        'fold': fold_number, 'size': size, 'stage': stage_name,
        'selected_lr': selected_lr, 'best_epoch': best_epoch,
        'best_holdout_dice': best_score, 'history': history,
        'lrf_lrs': lrf_lrs, 'lrf_losses': lrf_losses,
        'best_path': str(best_path),
    }
    atomic_torch_save({'complete': True, 'contract': contract, 'result': result}, last_path)
    del best_state, holdout_loader, optimizer, scaler, scheduler
    gc.collect()
    return result


In [ ]:

# ============================================================
# 1. KIỂM TRA GPU CÓ CHẠY ĐƯỢC VỚI BATCH SIZE ĐÃ CHỌN KHÔNG
# ============================================================

def preflight_batch(size):
    """
    Kiểm tra thử một batch trước khi bắt đầu huấn luyện chính thức.

    size:
        Kích thước ảnh đầu vào, ví dụ 128 hoặc 256.

    Mục đích:
        - Kiểm tra mô hình có chạy được với kích thước ảnh này không.
        - Kiểm tra Batch Size có vừa bộ nhớ GPU không.
        - Kiểm tra toàn bộ quy trình train_batch có hoạt động không.
    """

    # Lấy Batch Size tương ứng với kích thước ảnh.
    # Ví dụ:
    # size = 128 -> batch_size = 16
    # size = 256 -> batch_size = 8
    batch_size = BATCH_SIZE_BY_SIZE[size]

    # Khởi tạo mô hình MobileNetV2-UNet.
    # pretrained=False: không tải trọng số pretrained ImageNet,
    # vì đây chỉ là kiểm tra thử, không phải huấn luyện thật.
    #
    # to_runtime(model): chuyển mô hình lên DEVICE (thường là GPU)
    # và áp dụng định dạng bộ nhớ nếu được cấu hình.
    model = to_runtime(
        MobileNetV2UNet(pretrained=False)
    )

    # Tạo optimizer Adam với Learning Rate thử nghiệm = 0.0001.
    # Optimizer sẽ cập nhật trọng số trong lần train thử này.
    optimizer = optimizer_for(model, 1e-4)

    # Tạo GradScaler để hỗ trợ AMP nếu USE_AMP=True.
    scaler = make_scaler()

    # --------------------------------------------------------
    # TẠO DỮ LIỆU GIẢ ĐỂ KIỂM TRA
    # --------------------------------------------------------

    # Tạo batch ảnh RGB ngẫu nhiên.
    #
    # Kích thước tensor:
    # [Batch Size, 3, Height, Width]
    #
    # dtype=torch.uint8: mỗi pixel có giá trị nguyên từ 0 đến 255.
    #
    # Ví dụ size=128, batch_size=16:
    # inputs có shape [16, 3, 128, 128].
    inputs = torch.randint(
        0,
        256,
        (batch_size, 3, size, size),
        dtype=torch.uint8
    )

    # Tạo batch mask giả tương ứng với các ảnh.
    #
    # Kích thước:
    # [Batch Size, 1, Height, Width]
    #
    # Giá trị ngẫu nhiên:
    # - 0: nền.
    # - 1: vùng tổn thương.
    #
    # dtype=torch.float32: phù hợp với BCEWithLogitsLoss.
    targets = torch.randint(
        0,
        2,
        (batch_size, 1, size, size),
        dtype=torch.float32
    )

    try:
        # ----------------------------------------------------
        # CHẠY THỬ MỘT BƯỚC HUẤN LUYỆN
        # ----------------------------------------------------

        # train_batch thực hiện đầy đủ:
        # 1. Chuẩn hóa ảnh.
        # 2. Đưa ảnh qua mô hình (forward).
        # 3. Tính Loss.
        # 4. Tính gradient (backward).
        # 5. Cập nhật trọng số bằng optimizer.
        #
        # .cpu(): chuyển Loss về CPU.
        # float(): chuyển thành số thực Python để in kết quả.
        loss = float(
            train_batch(
                model,
                optimizer,
                scaler,
                inputs,
                targets
            ).cpu()
        )

        # Chờ GPU hoàn thành tất cả phép tính đang chờ.
        # Giúp phát hiện lỗi GPU phát sinh trong quá trình chạy.
        torch.cuda.synchronize()

    # --------------------------------------------------------
    # XỬ LÝ KHI GPU KHÔNG ĐỦ BỘ NHỚ
    # --------------------------------------------------------

    except torch.cuda.OutOfMemoryError as error:

        # Nếu batch không vừa GPU thì báo lỗi dễ hiểu hơn.
        # Gợi ý giảm Batch Size trong cấu hình.
        raise RuntimeError(
            f'Batch {batch_size} không vừa GPU ở {size}px; '
            'giảm PER_GPU_BATCH_BY_SIZE.'
        ) from error

    # --------------------------------------------------------
    # GIẢI PHÓNG BỘ NHỚ SAU KHI KIỂM TRA
    # --------------------------------------------------------

    finally:

        # Xóa các đối tượng chỉ dùng cho lần kiểm tra này.
        del model, optimizer, scaler, inputs, targets

        # Yêu cầu Python thu gom các đối tượng không còn sử dụng.
        gc.collect()

        # Giải phóng các vùng bộ nhớ GPU đang được PyTorch
        # giữ lại trong cache nhưng không còn cần thiết.
        torch.cuda.empty_cache()

    # In kết quả kiểm tra nếu không xảy ra lỗi.
    print(
        f'Preflight {size}px '
        f'batch={batch_size}: loss={loss:.4f}'
    )


# ============================================================
# 2. KIỂM TRA TẤT CẢ KÍCH THƯỚC ẢNH ĐÃ CẤU HÌNH
# ============================================================

# SIZES có thể là (128, 256).
# Vòng lặp lần lượt kiểm tra:
# - Ảnh 128x128 với Batch Size tương ứng.
# - Ảnh 256x256 với Batch Size tương ứng.
for size in SIZES:
    preflight_batch(size)


In [ ]:

# ============================================================
# 1. LƯU CẤU HÌNH CỦA TOÀN BỘ THÍ NGHIỆM
# ============================================================

# RUN_CONFIG lưu lại những thiết lập được sử dụng trong quá trình train.
# Mục đích:
# - Biết mô hình được huấn luyện với cấu hình nào.
# - Có thể kiểm tra và tái lập thí nghiệm.
# - Kiểm tra checkpoint có thuộc đúng lần chạy hiện tại không.

RUN_CONFIG = {

    # --------------------------------------------------------
    # THÔNG TIN DATASET
    # --------------------------------------------------------

    # Tên/định danh dataset trên Kaggle.
    'dataset_handle': DATASET_HANDLE,

    # Số lượng ảnh mong đợi trong từng tập.
    # Ví dụ: train = 2594, validation = 100, test = 1000.
    'expected_counts': EXPECTED_COUNTS,

    # Ba tập dữ liệu chính thức được sử dụng.
    'official_splits_used': ['Training', 'Validation', 'Test'],

    # Ghi nhận có đánh giá trên tập Validation và Test chính thức.
    'official_validation_evaluated': True,
    'official_test_evaluated': True,

    # --------------------------------------------------------
    # THIẾT LẬP CROSS-VALIDATION
    # --------------------------------------------------------

    # Seed dùng để kiểm soát tính ngẫu nhiên.
    'seed': SEED,

    # Số Fold trong Cross-Validation.
    'n_folds': N_FOLDS,

    # Các kích thước ảnh đầu vào, ví dụ 128 và 256.
    'sizes': list(SIZES),

    # Số epoch của hai giai đoạn huấn luyện.
    'frozen_epochs': FROZEN_EPOCHS,
    'unfrozen_epochs': UNFROZEN_EPOCHS,

    # Batch Size tương ứng với từng kích thước ảnh.
    'batch_size_by_size': BATCH_SIZE_BY_SIZE,

    # Số lượng DataLoader workers.
    'num_workers': NUM_WORKERS,

    # Số batch được chuẩn bị trước bởi mỗi worker.
    'prefetch_factor': PREFETCH_FACTOR,

    # Có sử dụng pinned memory khi nạp dữ liệu hay không.
    'pin_memory': PIN_MEMORY,

    # Có sử dụng trọng số pretrained hay không.
    'pretrained': PRETRAINED,

    # Tiêu chí chọn mô hình tốt nhất:
    # Dice trên tập Holdout nội bộ của từng Fold.
    'selection_metric': 'internal_fold_holdout_dice',

    # --------------------------------------------------------
    # LOSS VÀ OPTIMIZER
    # --------------------------------------------------------

    # Hàm mất mát sử dụng.
    'loss': 'BCEWithLogitsLoss',

    # Optimizer sử dụng.
    # Nếu USE_FUSED_ADAM=True thì dùng Adam fused.
    'optimizer': 'Adam(fused)' if USE_FUSED_ADAM else 'Adam',

    # Weight Decay: hệ số regularization.
    'weight_decay': WEIGHT_DECAY,

    # Giới hạn độ lớn gradient.
    'grad_clip_norm': GRAD_CLIP_NORM,

    # --------------------------------------------------------
    # CẤU HÌNH LR FINDER
    # --------------------------------------------------------

    # Các thông số dùng để tìm Learning Rate phù hợp.
    'lr_finder': {
        'min': LR_MIN,
        'max': LR_MAX,
        'iterations': LRF_ITERS,
        'beta': LRF_BETA,
        'selection_factor': LR_SELECTION_FACTOR,
        'max_lr_frozen': MAX_LR_FROZEN,
        'max_lr_unfrozen': MAX_LR_UNFROZEN
    },

    # Các thông số của lịch điều chỉnh Learning Rate STLR.
    'stlr': {
        'cut_frac': CUT_FRAC,
        'ratio': STLR_RATIO
    },

    # --------------------------------------------------------
    # DATA AUGMENTATION VÀ PREPROCESSING
    # --------------------------------------------------------

    # Các phép biến đổi ảnh khi huấn luyện:
    # D4 + Affine + BrightnessContrast.
    'augmentation': 'D4+Affine(rotate44,scale1.0-1.05)+BrightnessContrast0.2',

    # Chỉ áp dụng augmentation cho tập train.
    'augmentation_only_on_train_subset': True,

    # Các bước preprocessing:
    # Resize -> Gray World -> ImageNet Normalize trên GPU.
    'preprocessing': 'resize+gray_world+imagenet_normalize_on_gpu',

    # --------------------------------------------------------
    # CẤU HÌNH TỐI ƯU GPU VÀ BỘ NHỚ
    # --------------------------------------------------------

    # Có sử dụng Automatic Mixed Precision không?
    'amp': USE_AMP,

    # Có sử dụng memory format channels_last không?
    'channels_last': USE_CHANNELS_LAST,

    # Có sử dụng FP32 cho Transposed Convolution không?
    'fp32_transposed_conv': FP32_TRANSPOSED_CONV,

    # Có sử dụng Fused Adam không?
    'fused_adam': USE_FUSED_ADAM,

    # Có lưu ảnh train đã resize trong RAM không?
    'cache_train_uint8': CACHE_RESIZED_IN_RAM,

    # Có yêu cầu hệ điều hành giải phóng page cache không?
    'drop_os_page_cache': DROP_OS_PAGE_CACHE,

    # Cứ bao nhiêu epoch thì lưu checkpoint gần nhất một lần.
    'save_last_every': SAVE_LAST_EVERY,

    # --------------------------------------------------------
    # THIẾT LẬP ĐÁNH GIÁ
    # --------------------------------------------------------

    # Ngưỡng chuyển xác suất thành mask nhị phân.
    'prediction_threshold': PRED_THRESHOLD,

    # Ngưỡng Jaccard dùng trong metric threshold_jaccard.
    'jaccard_cut': JACCARD_CUT,

    # Phiên bản protocol của thí nghiệm.
    'protocol_version': PROTOCOL_VERSION,
}


# Ghi cấu hình vào file config.json trong RUN_DIR.
#
# indent=2: trình bày JSON dễ đọc.
# ensure_ascii=False: giữ nguyên ký tự tiếng Việt nếu có.
# encoding='utf-8': sử dụng mã hóa UTF-8.
(RUN_DIR / 'config.json').write_text(
    json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False),
    encoding='utf-8'
)


# ============================================================
# 2. HUẤN LUYỆN MỘT FOLD
# ============================================================

def train_fold(fold_number, train_subset, holdout_subset):
    """
    Huấn luyện mô hình trên một Fold.

    fold_number:
        Số thứ tự Fold hiện tại.

    train_subset:
        Phần dữ liệu dùng để huấn luyện.

    holdout_subset:
        Phần dữ liệu dùng để đánh giá và chọn mô hình tốt nhất.

    Mỗi Fold sẽ trải qua:
        1. Frozen stage.
        2. Unfrozen stage.

    Với mỗi kích thước ảnh trong SIZES, hai giai đoạn trên
    đều được thực hiện.
    """

    # Tạo seed riêng cho từng Fold.
    # Ví dụ:
    # Fold 1 -> SEED
    # Fold 2 -> SEED + 1
    # Fold 3 -> SEED + 2
    fold_seed = SEED + fold_number - 1

    # Đặt seed để kiểm soát tính ngẫu nhiên trong Fold này.
    set_seed(fold_seed)

    # Tạo đường dẫn đến checkpoint của giai đoạn đầu tiên.
    # Ví dụ: fold_1/128px_frozen/last.pt
    first_marker = (
        CHECKPOINT_DIR
        / f'fold_{fold_number}'
        / f'{SIZES[0]}px_frozen'
        / 'last.pt'
    )

    # Kiểm tra có đang tiếp tục một Fold đã chạy dở không.
    # Nếu RESUME=True và checkpoint đầu tiên tồn tại
    # thì continuing=True.
    continuing = RESUME and first_marker.is_file()

    # Khởi tạo MobileNetV2-UNet và chuyển lên thiết bị tính toán.
    #
    # pretrained=PRETRAINED and not continuing:
    # - Nếu là lần chạy mới: có thể sử dụng pretrained.
    # - Nếu đang resume: không cần tải pretrained lại,
    #   vì trọng số sẽ được khôi phục từ checkpoint.
    model = to_runtime(
        MobileNetV2UNet(
            pretrained=PRETRAINED and not continuing
        )
    )

    # Danh sách lưu kết quả của từng stage.
    stage_results = []

    # --------------------------------------------------------
    # 2.1. DUYỆT QUA CÁC KÍCH THƯỚC ẢNH
    # --------------------------------------------------------

    for size_index, size in enumerate(SIZES):

        # ====================================================
        # GIAI ĐOẠN 1: FROZEN
        # ====================================================

        # Đóng băng phần encoder theo quy tắc đã định nghĩa
        # trong freeze_first_layer_group().
        # Decoder vẫn được phép học.
        freeze_first_layer_group(model)

        # Huấn luyện ở giai đoạn Frozen.
        stage_results.append(
            run_stage(
                model,
                train_subset,
                holdout_subset,
                size,
                FROZEN_EPOCHS,

                # freeze_bn=False:
                # Không yêu cầu giữ BatchNorm ở eval trong stage này.
                False,

                fold_number,
                'frozen',

                # Seed riêng cho từng kích thước ảnh và stage.
                fold_seed + size_index * 1000,
            )
        )

        # ====================================================
        # GIAI ĐOẠN 2: UNFROZEN
        # ====================================================

        # Mở trainable parameters của encoder,
        # đồng thời giữ BatchNorm ở trạng thái đóng băng
        # theo logic của unfreeze_with_frozen_bn().
        unfreeze_with_frozen_bn(model)

        # Tiếp tục huấn luyện ở giai đoạn Unfrozen.
        stage_results.append(
            run_stage(
                model,
                train_subset,
                holdout_subset,
                size,
                UNFROZEN_EPOCHS,

                # freeze_bn=True:
                # Giữ BatchNorm ở chế độ eval trong quá trình train.
                True,

                fold_number,
                'unfrozen',

                # Seed riêng cho stage Unfrozen.
                fold_seed + size_index * 1000 + 500,
            )
        )

    # --------------------------------------------------------
    # 2.2. ĐÁNH GIÁ HOLDOUT SAU KHI HOÀN THÀNH FOLD
    # --------------------------------------------------------

    # Tạo DataLoader cho holdout.
    # SIZES[-1] là kích thước ảnh cuối cùng trong SIZES.
    holdout_loader = make_loader(
        holdout_subset,
        SIZES[-1],
        train=False,
        seed=fold_seed
    )

    # Tính Dice của mô hình sau khi hoàn thành toàn bộ stage.
    final_holdout_dice = evaluate_dice(
        model,
        holdout_loader
    )

    # Đường dẫn lưu mô hình tốt nhất cuối cùng của Fold.
    # Ví dụ: fold_1/mobilenetv2_unet_best.pt
    final_path = (
        CHECKPOINT_DIR
        / f'fold_{fold_number}'
        / 'mobilenetv2_unet_best.pt'
    )

    # --------------------------------------------------------
    # 2.3. LƯU METADATA CỦA FOLD
    # --------------------------------------------------------

    # Metadata lưu thông tin về Fold:
    # - Seed.
    # - Cấu hình.
    # - Số lượng ảnh train/holdout.
    # - Danh sách ID ảnh.
    # - Holdout Dice.
    # - Kết quả từng stage.
    metadata = {
        'fold': fold_number,
        'fold_seed': fold_seed,
        'config': RUN_CONFIG,

        'train_count': len(train_subset),
        'holdout_count': len(holdout_subset),

        # Mã băm ID ảnh để xác định chính xác split.
        'train_ids': ids_digest(train_subset),
        'holdout_ids': ids_digest(holdout_subset),

        # Dice trên holdout sau khi hoàn thành Fold.
        'holdout_dice': final_holdout_dice,

        # Kết quả của tất cả stage trong Fold.
        'stage_results': stage_results,
    }

    # Lưu trọng số mô hình cuối cùng của Fold cùng metadata.
    atomic_torch_save(
        {
            'model_state': unwrap_model(model).state_dict(),
            'meta': metadata
        },
        final_path
    )

    # --------------------------------------------------------
    # 2.4. LƯU LỊCH SỬ HUẤN LUYỆN RA CSV
    # --------------------------------------------------------

    # Gộp history của tất cả stage thành một danh sách.
    #
    # Mỗi result trong stage_results có một danh sách history.
    rows = [
        row
        for result in stage_results
        for row in result['history']
    ]

    # Chuyển history thành DataFrame và lưu thành CSV.
    # File giúp theo dõi Loss, Holdout Dice, Learning Rate...
    pd.DataFrame(rows).to_csv(
        LOG_DIR / f'fold_{fold_number}_history.csv',
        index=False
    )

    # --------------------------------------------------------
    # 2.5. GIẢI PHÓNG BỘ NHỚ
    # --------------------------------------------------------

    # Xóa mô hình và holdout loader sau khi hoàn thành Fold.
    del model, holdout_loader

    # Thu gom các đối tượng Python không còn sử dụng.
    gc.collect()

    # Giải phóng bộ nhớ GPU đang được giữ trong cache.
    torch.cuda.empty_cache()

    # Trả về:
    # 1. Đường dẫn checkpoint của Fold.
    # 2. Metadata của Fold.
    return final_path, metadata


# ============================================================
# 3. CHẠY TOÀN BỘ CÁC FOLD
# ============================================================

# Danh sách lưu đường dẫn checkpoint của từng Fold.
fold_checkpoint_paths = []

# Danh sách lưu metadata của từng Fold.
fold_metadata = []

# Ghi lại thời điểm bắt đầu huấn luyện toàn bộ các Fold.
training_started = time.time()


# ------------------------------------------------------------
# 3.1. DUYỆT QUA TỪNG FOLD
# ------------------------------------------------------------

# folds chứa các cặp:
# (train_subset, holdout_subset).
#
# enumerate(..., start=1):
# Đánh số Fold từ 1 thay vì 0.
for fold_number, (train_subset, holdout_subset) in enumerate(
    folds,
    start=1
):

    # Đường dẫn checkpoint hoàn chỉnh của Fold hiện tại.
    final_path = (
        CHECKPOINT_DIR
        / f'fold_{fold_number}'
        / 'mobilenetv2_unet_best.pt'
    )

    # --------------------------------------------------------
    # 3.2. KIỂM TRA FOLD ĐÃ HOÀN THÀNH TRƯỚC ĐÓ CHƯA
    # --------------------------------------------------------

    if RESUME and final_path.is_file():

        # Nếu checkpoint cuối đã tồn tại và RESUME bật,
        # đọc metadata để kiểm tra có thể sử dụng lại không.
        payload = trusted_torch_load(
            final_path,
            map_location='cpu'
        )

        metadata = payload['meta']

        # Kiểm tra:
        # 1. Cấu hình có giống lần chạy hiện tại không.
        # 2. Danh sách train ID có giống không.
        # 3. Danh sách holdout ID có giống không.
        #
        # Nếu khác thì không sử dụng checkpoint cũ.
        if (
            metadata.get('config') != RUN_CONFIG
            or metadata.get('train_ids') != ids_digest(train_subset)
            or metadata.get('holdout_ids') != ids_digest(holdout_subset)
        ):
            raise RuntimeError(
                f'Fold checkpoint không khớp: {final_path}'
            )

        # Không cần giữ payload trong RAM nữa.
        del payload

        # Sử dụng lại checkpoint đã hoàn thành.
        fold_path = final_path

        # In thông báo Fold đã hoàn thành từ trước.
        print(
            f"[resume] fold {fold_number} complete, "
            f"Dice={metadata['holdout_dice']:.4f}"
        )

    # --------------------------------------------------------
    # 3.3. NẾU FOLD CHƯA HOÀN THÀNH THÌ HUẤN LUYỆN
    # --------------------------------------------------------

    else:

        # In số lượng ảnh train và holdout của Fold.
        print(
            f'\n===== FOLD {fold_number}/{N_FOLDS}: '
            f'train={len(train_subset)}, '
            f'holdout={len(holdout_subset)} ====='
        )

        # Gọi train_fold() để thực hiện toàn bộ quá trình:
        # Frozen -> Unfrozen -> đánh giá -> lưu checkpoint.
        fold_path, metadata = train_fold(
            fold_number,
            train_subset,
            holdout_subset
        )

    # --------------------------------------------------------
    # 3.4. GHI NHẬN KẾT QUẢ CỦA FOLD
    # --------------------------------------------------------

    # Lưu đường dẫn checkpoint của Fold.
    fold_checkpoint_paths.append(str(fold_path))

    # Lưu metadata của Fold.
    fold_metadata.append(metadata)


# ============================================================
# 4. TỔNG HỢP KẾT QUẢ HOLDOUT CỦA CÁC FOLD
# ============================================================

# Tạo DataFrame từ metadata của từng Fold.
#
# Mỗi hàng tương ứng với một Fold.
# Các cột gồm:
# - fold: số thứ tự Fold.
# - train_count: số ảnh train.
# - holdout_count: số ảnh holdout.
# - internal_holdout_dice: Dice trên holdout.
fold_table = pd.DataFrame([
    {
        'fold': metadata['fold'],
        'train_count': metadata['train_count'],
        'holdout_count': metadata['holdout_count'],
        'internal_holdout_dice': metadata['holdout_dice']
    }
    for metadata in fold_metadata
])


# ------------------------------------------------------------
# 4.1. TÍNH THỜI GIAN HUẤN LUYỆN
# ------------------------------------------------------------

# Lấy thời gian kết thúc trừ thời gian bắt đầu.
# Chia 3600 để chuyển từ giây sang giờ.
training_elapsed_hours = (
    time.time() - training_started
) / 3600


# ------------------------------------------------------------
# 4.2. LƯU BẢNG KẾT QUẢ
# ------------------------------------------------------------

# Lưu bảng kết quả Holdout Dice của các Fold thành CSV.
fold_table.to_csv(
    LOG_DIR / 'fold_holdout_dice.csv',
    index=False
)

# Hiển thị bảng kết quả trong Notebook.
# format giúp Dice được hiển thị với 4 chữ số thập phân.
display(
    fold_table.style.format({
        'internal_holdout_dice': '{:.4f}'
    })
)


# ------------------------------------------------------------
# 4.3. THÔNG BÁO HOÀN THÀNH HUẤN LUYỆN
# ------------------------------------------------------------

# In tổng thời gian huấn luyện.
# Đồng thời thông báo cell tiếp theo sẽ đánh giá
# trên Validation/Test chính thức.
print(
    f'Training complete in {training_elapsed_hours:.2f} hours. '
    'Starting official validation/test in the next cell.'
)

# In danh sách đường dẫn checkpoint của các Fold.
print('Checkpoints:', fold_checkpoint_paths)


## Đánh giá official validation và test

Single model được khóa bằng internal fold-holdout Dice trước khi mở official validation/test. Validation chỉ dùng để báo cáo từng fold và equal-weight probability ensemble; test báo cáo single đã khóa và ensemble. HD95 có đơn vị pixel trên mask đã resize `256×256`.

In [ ]:
# ============================================================
# BƯỚC 1: CHỌN SINGLE BEST MODEL VÀ NẠP CÁC MÔ HÌNH
# ============================================================

# Lấy holdout_dice của cả 3 Fold rồi tìm Fold có Dice cao nhất.
# np.argmax() trả về vị trí (index) có giá trị lớn nhất,
# index bắt đầu từ 0.
selected_single_index = int(
    np.argmax([meta['holdout_dice'] for meta in fold_metadata])
)

# Chuyển index thành số Fold bắt đầu từ 1.
# Ví dụ: index = 1 thì tương ứng Fold 2.
selected_single_fold = selected_single_index + 1

# Đường dẫn lưu checkpoint riêng của mô hình đơn tốt nhất.
selected_checkpoint_path = (
    CHECKPOINT_DIR / 'mobilenetv2_unet_selected_single_best.pt'
)

# Danh sách lưu 3 mô hình đã được nạp lên GPU.
# Sau này dùng danh sách này để đánh giá từng mô hình
# và kết hợp dự đoán bằng Ensemble.
runtime_models = []

# Duyệt qua đường dẫn checkpoint của từng Fold.
for index, checkpoint_path in enumerate(fold_checkpoint_paths):

    # Đọc checkpoint lên CPU.
    # Checkpoint chứa trọng số mô hình và metadata.
    payload = trusted_torch_load(
        checkpoint_path,
        map_location='cpu'
    )

    # Kiểm tra xem mô hình hiện tại có phải Fold tốt nhất không.
    if index == selected_single_index:

        # Sao chép metadata của Fold được chọn.
        selected_meta = dict(payload['meta'])

        # Ghi lại lý do lựa chọn mô hình.
        selected_meta['selection'] = (
            'highest internal fold-holdout Dice'
        )

        # Ghi số Fold được chọn.
        selected_meta['selected_fold'] = selected_single_fold

        # Lưu riêng trọng số và metadata của mô hình đơn tốt nhất.
        atomic_torch_save(
            {
                'model_state': payload['model_state'],
                'meta': selected_meta
            },
            selected_checkpoint_path
        )

    # Khởi tạo một mô hình MobileNetV2-UNet mới.
    # pretrained=False vì ta sẽ nạp trọng số đã train từ checkpoint.
    model = to_runtime(
        MobileNetV2UNet(pretrained=False)
    )

    # Nạp trọng số đã huấn luyện của Fold hiện tại vào mô hình.
    load_model_state(model, payload['model_state'])

    # Thêm mô hình vào danh sách.
    # Sau vòng lặp, runtime_models sẽ chứa đủ 3 mô hình.
    runtime_models.append(model)

    # Giải phóng checkpoint tạm trên CPU.
    del payload

# Thu hồi bộ nhớ RAM không còn sử dụng.
gc.collect()

# Giải phóng phần VRAM PyTorch đang giữ nhưng không còn dùng.
torch.cuda.empty_cache()

# ============================================================
# BƯỚC 2: ĐÁNH GIÁ TRÊN OFFICIAL VALIDATION
# ============================================================

# Tạo DataLoader cho tập Validation chính thức.
# SIZES[-1] lấy kích thước ảnh cuối cùng trong SIZES.
# Ví dụ SIZES = (128, 256) thì sử dụng ảnh 256x256.
# train=False: không áp dụng augmentation.
validation_loader = make_loader(
    official_val_records,
    SIZES[-1],
    train=False,
    seed=SEED
)

# Danh sách lưu kết quả đánh giá của Validation.
validation_rows = []

# ------------------------------------------------------------
# 2.1. Đánh giá riêng từng mô hình Fold
# ------------------------------------------------------------

# Duyệt qua 3 mô hình trong runtime_models.
# start=1 để số thứ tự hiển thị bắt đầu từ Fold 1.
for index, model in enumerate(runtime_models, start=1):

    validation_rows.append({

        # Tên mô hình trong bảng kết quả.
        'model': f'BestDice fold {index}',

        # Đánh giá mô hình hiện tại trên Validation.
        # [model] là danh sách chỉ chứa một mô hình.
        # ** dùng để đưa các chỉ số trả về từ evaluate_full()
        # vào dictionary hiện tại.
        #
        # include_hd95=True: tính thêm chỉ số HD95.
        **evaluate_full(
            [model],
            validation_loader,
            include_hd95=True
        ),
    })

# ------------------------------------------------------------
# 2.2. Đánh giá Ensemble
# ------------------------------------------------------------

validation_rows.append({

    # Tên phương pháp đánh giá.
    'model': 'Equal-weight ensemble',

    # Đưa cả 3 mô hình vào evaluate_full().
    #
    # Bên trong evaluate_full(), các mô hình sẽ được kết hợp
    # dự đoán bằng ensemble_probability().
    #
    # Equal-weight nghĩa là mỗi mô hình có trọng số bằng nhau.
    **evaluate_full(
        runtime_models,
        validation_loader,
        include_hd95=True
    ),
})

# Chuyển danh sách kết quả thành DataFrame để dễ xem.
validation_table = pd.DataFrame(validation_rows)

# Lưu bảng kết quả Validation thành file CSV.
validation_table.to_csv(
    LOG_DIR / 'official_validation_metrics.csv',
    index=False
)

# Đánh giá xong thì xóa DataLoader để giải phóng bộ nhớ.
del validation_loader

# ============================================================
# BƯỚC 3: ĐÁNH GIÁ TRÊN OFFICIAL TEST
# ============================================================

# Tạo DataLoader cho tập Test chính thức.
# Không sử dụng augmentation.
test_loader = make_loader(
    test_records,
    SIZES[-1],
    train=False,
    seed=SEED
)

# Danh sách lưu kết quả đánh giá trên Test.
test_rows = [

    # --------------------------------------------------------
    # 3.1. Đánh giá Single Best Model
    # --------------------------------------------------------
    {
        # Tên mô hình được chọn ở Bước 1.
        'model': f'Single BestDice fold {selected_single_fold}',

        # Chỉ đánh giá mô hình có holdout_dice cao nhất.
        # selected_single_index xác định vị trí mô hình
        # tương ứng trong runtime_models.
        **evaluate_full(
            [runtime_models[selected_single_index]],
            test_loader,
            include_hd95=True
        )
    },

    # --------------------------------------------------------
    # 3.2. Đánh giá Ensemble
    # --------------------------------------------------------
    {
        'model': 'Equal-weight ensemble',

        # Kết hợp dự đoán từ cả 3 Fold rồi đánh giá.
        **evaluate_full(
            runtime_models,
            test_loader,
            include_hd95=True
        )
    },
]

# Chuyển kết quả thành DataFrame.
test_table = pd.DataFrame(test_rows)

# Lưu kết quả Test thành CSV.
test_table.to_csv(
    LOG_DIR / 'official_test_metrics.csv',
    index=False
)

# Đánh giá xong thì xóa DataLoader.
del test_loader

# ============================================================
# BƯỚC 4: TỔNG HỢP VÀ LƯU KẾT QUẢ
# ============================================================

# Tạo dictionary chứa thông tin của toàn bộ quá trình.
summary = {

    # Lưu cấu hình đã sử dụng khi huấn luyện.
    'config': RUN_CONFIG,

    # Số lượng ảnh trong từng tập dữ liệu.
    'sample_counts': {
        'train': len(train_records),
        'validation': len(official_val_records),
        'test': len(test_records)
    },

    # Tổng thời gian huấn luyện tính bằng giờ.
    'elapsed_training_hours': training_elapsed_hours,

    # Đường dẫn checkpoint của 3 Fold.
    'fold_checkpoints': fold_checkpoint_paths,

    # Kết quả Internal Holdout Dice của từng Fold.
    # orient='records' chuyển DataFrame thành danh sách dictionary.
    'fold_internal_holdout': fold_table.to_dict(
        orient='records'
    ),

    # Số Fold được chọn làm Single Best Model.
    'selected_single_fold_by_internal_cv_dice': selected_single_fold,

    # Đường dẫn checkpoint của Single Best Model.
    'selected_single_checkpoint': str(selected_checkpoint_path),

    # Kết quả đánh giá trên Validation.
    'official_validation': validation_table.to_dict(
        orient='records'
    ),

    # Kết quả đánh giá trên Test.
    'official_test': test_table.to_dict(
        orient='records'
    ),

    # Ghi chú đơn vị của HD95.
    # Ở đây HD95 được tính bằng pixel tại độ phân giải cuối cùng.
    'hd95_unit': f'pixel at {SIZES[-1]}x{SIZES[-1]} resolution',
}

# Xác định đường dẫn lưu file summary.json.
summary_path = RUN_DIR / 'summary.json'

# Chuyển dictionary thành JSON và lưu vào ổ đĩa.
# indent=2 giúp JSON dễ đọc.
# ensure_ascii=False giữ nguyên ký tự tiếng Việt nếu có.
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding='utf-8'
)

# ------------------------------------------------------------
# 4.1. Hiển thị bảng Validation
# ------------------------------------------------------------

# Định dạng các chỉ số thành 4 chữ số thập phân.
# Không định dạng cột model vì đây là tên mô hình.
metric_format = {
    column: '{:.4f}'
    for column in validation_table.columns
    if column != 'model'
}

# Hiển thị kết quả Validation.
display(
    validation_table.style.format(metric_format)
)

# ------------------------------------------------------------
# 4.2. Hiển thị bảng Test
# ------------------------------------------------------------

# Hiển thị kết quả Test với 4 chữ số thập phân.
display(
    test_table.style.format({
        column: '{:.4f}'
        for column in test_table.columns
        if column != 'model'
    })
)

# ------------------------------------------------------------
# 4.3. In thông tin quan trọng
# ------------------------------------------------------------

# In Fold được chọn làm Single Best Model.
print('Selected single fold:', selected_single_fold)

# In đường dẫn checkpoint của mô hình đơn tốt nhất.
print('Selected checkpoint:', selected_checkpoint_path)

# In đường dẫn file tổng hợp kết quả.
print('Summary:', summary_path)

## Kết quả được tạo

- Ba checkpoint fold-final để resume/tái tạo ensemble và một checkpoint single được chọn hoàn toàn bằng internal CV Dice.
- Lịch sử train từng fold, bảng internal holdout, official validation và official test trong `logs`.
- `summary.json` chứa cấu hình, split counts, checkpoint paths và toàn bộ metrics.
- Official validation/test không tham gia LR Finder, augmentation, early stopping, chọn fold, threshold hoặc ensemble weight.

In [ ]:
# from pathlib import Path

# root = Path(
#     "/kaggle/working/mobilenetv2_unet_kaggle_1gpu/full_v3"
# )

# # Kiểm tra các checkpoint
# checkpoint_dir = root / "checkpoints"

# print("Checkpoint directory exists:", checkpoint_dir.exists())

# if checkpoint_dir.exists():
#     checkpoints = list(checkpoint_dir.rglob("*.pt")) + \
#                   list(checkpoint_dir.rglob("*.pth"))

#     if checkpoints:
#         print("\nĐÃ TÌM THẤY CHECKPOINT:")
#         for p in checkpoints:
#             print(f"- {p}")
#             print(f"  Size: {p.stat().st_size / (1024**2):.2f} MB")
#     else:
#         print("Chưa tìm thấy file .pt hoặc .pth")

# # Kiểm tra file tổng kết
# print("\nsummary.json exists:", (root / "summary.json").exists())

In [ ]:
# from pathlib import Path

# root = Path("/kaggle/working")
# patterns = ["*.pt", "*.pth", "summary.json", "config.json"]

# found = []

# for pattern in patterns:
#     found.extend(root.rglob(pattern))

# if found:
#     for f in sorted(set(found)):
#         print(f"{f} | {f.stat().st_size / 1024**2:.2f} MB")
# else:
#     print("Không tìm thấy checkpoint hoặc file kết quả trong /kaggle/working/")

In [ ]:
# from pathlib import Path

# print("Current working directory:")
# print(Path.cwd())

# print("\nKaggle working directory:")
# print(list(Path("/kaggle/working").iterdir()))

# print("\nCheckpoint files anywhere under /kaggle:")
# for p in Path("/kaggle").rglob("mobilenetv2_unet_selected_single_best.pt"):
#     print(p)